In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
import sys

!{sys.executable} -m pip install -q kagglehub evaluate wandb tensorboard bitsandbytes accelerate peft transformers torch scikit-learn tqdm matplotlib seaborn
!pip install --upgrade transformers
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import psutil
import warnings
warnings.filterwarnings('ignore')


print("All libraries installed successfully!")


All libraries installed successfully!


# Dataset Download and processing

In [3]:
import kagglehub

path = kagglehub.dataset_download("peiyuanliu2001/mmlu-dataset")

print("Dataset path:", path)

Dataset path: /root/.cache/kagglehub/datasets/peiyuanliu2001/mmlu-dataset/versions/2


In [4]:
import pandas as pd
from datasets import Dataset

# Load the dataset
df = pd.read_csv(f"{path}/train.csv")

print("Dataset Overview:")
print(f"  Total samples: {len(df):,}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\nAnswer distribution:")
print(df['answer'].value_counts().to_dict())

print("\nSample questions:")
display(df.head())

Dataset Overview:
  Total samples: 98,487
  Columns: ['prompt', 'A', 'B', 'C', 'D', 'answer']

Answer distribution:
{'C': 26491, 'B': 25377, 'D': 24767, 'A': 21852}

Sample questions:


,prompt,A,B,C,D,answer
0,Davis decided to kill Adams. He set out for Ad...,Adams only.,Brooks only.,Case only.,Adams and Brooks,B
1,A state statute requires any person licensed t...,"guilty, because this is a public welfare offen...","guilty, because he cannot be excused on the ba...","not guilty, because the statute punishes omiss...","not guilty, because he was not aware of the va...",D
2,"Lender met Borrower on the street, demanded th...","Yes, because Mann threatened to use deadly for...","Yes, unless Mann was related to Borrower.","No, if it was apparent that Lender was about t...","No, because Lender was the original aggressor ...",C
3,Peter sued Don for breach of contract. The cou...,must permit Don to answer if he had objected t...,"may permit Don to answer, whether or not he ha...",may permit Don to answer only if he had object...,"cannot permit Don to answer, whether or not he...",B
4,Ames had painted Bell's house under a contract...,partial breach of contract only if Ames had pr...,partial breach of contract whether or not Ames...,total breach of contract only if Ames had prop...,total breach of contract whether or not Ames h...,C


In [5]:
from sklearn.model_selection import train_test_split

def format_mmlu_prompt(row):
    """Format MMLU questions for evaluation."""
    prompt = f"""Question: {row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}

Answer:"""
    return prompt

stratify_labels = df['subject'] if 'subject' in df.columns else None
train_indices, eval_indices = train_test_split(
    df.index,
    test_size=0.3,
    random_state=42,
    stratify=stratify_labels if stratify_labels is not None else None,
)

df['split'] = 'train'
df.loc[eval_indices, 'split'] = 'eval'
train_subset_for_examples = df.loc[train_indices].copy()

df['formatted_prompt'] = df.apply(format_mmlu_prompt, axis=1)

answer_to_idx = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
df['answer_idx'] = df['answer'].map(answer_to_idx)

print("\nSample formatted prompt:")

sample_prompt = df['formatted_prompt'].iloc[0]
print(sample_prompt[:500] + "..." if len(sample_prompt) > 500 else sample_prompt)

print(f"\nExpected answer: {df['answer'].iloc[0]}")
print(f"rompt length: {len(sample_prompt)} characters")

train_df = df[df['split'] == 'train'].copy()
eval_df = df[df['split'] == 'eval'].copy()

print(f"\nSplit summary:")
print(f"  Train samples: {len(train_df):,}")
print(f"  Eval samples:  {len(eval_df):,}")



Sample formatted prompt:
Question: Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were

A. Adams only.
B. Brooks only.
C. Case only.
D. Adams and Brooks

Answer...

Expected answer: B
rompt length: 501 characters

Split summary:
  Train samples: 68,940
  Eval samples:  29,547


In [8]:
from datasets import Dataset, DatasetDict


MAX_LENGTH = 512 
# Determine columns to include
columns_to_use = ['prompt', 'formatted_prompt', 'answer', 'answer_idx']
if 'subject' in train_df.columns:
    columns_to_use.insert(0, 'subject')
if 'A' in train_df.columns:
    columns_to_use.extend(['A', 'B', 'C', 'D'])

train_dataset_raw = Dataset.from_pandas(
    train_df[columns_to_use],
    preserve_index=False,
)
eval_dataset_raw = Dataset.from_pandas(
    eval_df[columns_to_use],
    preserve_index=False,
)

dataset = DatasetDict({
    'train': train_dataset_raw,
    'test': eval_dataset_raw,
})
eval_dataset = eval_dataset_raw

print("Dataset prepared:")
print(f"  Training samples: {len(train_dataset_raw):,}")
print(f"  Evaluation samples: {len(eval_dataset_raw):,}")
print(f"  Columns: {eval_dataset_raw.column_names}")


Dataset prepared:
  Training samples: 68,940
  Evaluation samples: 29,547
  Columns: ['prompt', 'formatted_prompt', 'answer', 'answer_idx', 'A', 'B', 'C', 'D']


# Model loading (8-bit quantized)

In [9]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Define the model path
model_id = "mistralai/Mistral-7B-v0.1"


bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_compute_dtype=torch.bfloat16,  
    bnb_8bit_quant_type="nf4", 
    llm_int8_enable_fp32_cpu_offload=True 
)

device_map = {"": "cuda:0"}

print("Loading model with 8-bit quantization (QLoRA)...")


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True
)

base_model = model

Loading model with 8-bit quantization (QLoRA)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.09s/it]


Tokenizer


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

In [11]:
# Get token IDs for answer choices
ANSWER_TOKENS = {
    'A': tokenizer.encode('A', add_special_tokens=False)[0],
    'B': tokenizer.encode('B', add_special_tokens=False)[0],
    'C': tokenizer.encode('C', add_special_tokens=False)[0],
    'D': tokenizer.encode('D', add_special_tokens=False)[0],
}

print("Answer token IDs:")
for letter, token_id in ANSWER_TOKENS.items():
    decoded = tokenizer.decode([token_id])
    print(f"  {letter}: {token_id} → '{decoded}'")

# Also create reverse mapping
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}



Answer token IDs:
  A: 330 → 'A'
  B: 365 → 'B'
  C: 334 → 'C'
  D: 384 → 'D'


# Evaluation functions

In [12]:
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import time

def compute_ece(confidences, predictions, labels, n_bins=10):
    """
    Compute Expected Calibration Error (ECE).
    Measures how well model confidence aligns with actual accuracy.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = in_bin.mean()

        if prop_in_bin > 0:
            accuracy_in_bin = (predictions[in_bin] == labels[in_bin]).mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return ece

def evaluate_mmlu_comprehensive(model, tokenizer, eval_dataset, answer_tokens,
                                 device="cuda", max_samples=None, show_progress=True):
    """
    Comprehensive MMLU evaluation with all Priority 1-3 metrics.

    Returns dict with:
    - accuracy: Overall accuracy
    - top2_accuracy: Top-2 accuracy
    - confidences: Model confidence scores
    - predictions: Predicted answers
    - true_labels: Ground truth answers
    - confusion_matrix: Confusion matrix
    - ece: Expected Calibration Error
    - throughput: Samples per second
    - avg_latency: Average latency per sample
    """
    model.eval()

    correct = 0
    top2_correct = 0
    total = 0

    all_confidences = []
    all_predictions = []
    all_true_labels = []

    # Limit samples if specified
    samples = eval_dataset
    if max_samples is not None:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    # Get token IDs as tensor
    answer_token_ids = torch.tensor([answer_tokens['A'], answer_tokens['B'],
                                      answer_tokens['C'], answer_tokens['D']])

    # Start timing
    start_time = time.time()

    iterator = tqdm(samples, desc="Evaluating", disable=not show_progress)

    with torch.no_grad():
        for example in iterator:
            prompt = example['formatted_prompt']
            true_answer = example['answer']
            true_idx = answer_to_idx[true_answer]

            # Tokenize and get logits
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            outputs = model(**inputs)
            last_logits = outputs.logits[0, -1, :]

            # Get probabilities for A, B, C, D
            answer_logits = last_logits[answer_token_ids]
            answer_probs = torch.softmax(answer_logits, dim=0)

            # Top-1 prediction
            pred_idx = answer_probs.argmax().item()
            pred_answer = idx_to_letter[pred_idx]
            confidence = answer_probs[pred_idx].item()

            # Top-2 prediction
            top2_indices = answer_probs.topk(2).indices.tolist()

            # Store results
            all_confidences.append(confidence)
            all_predictions.append(pred_idx)
            all_true_labels.append(true_idx)

            if pred_answer == true_answer:
                correct += 1
            if true_idx in top2_indices:
                top2_correct += 1
            total += 1

    # End timing
    end_time = time.time()
    duration = end_time - start_time

    # Convert to numpy arrays
    confidences = np.array(all_confidences)
    predictions = np.array(all_predictions)
    true_labels = np.array(all_true_labels)

    # Compute metrics
    accuracy = correct / total if total > 0 else 0.0
    top2_accuracy = top2_correct / total if total > 0 else 0.0
    ece = compute_ece(confidences, predictions, true_labels)
    conf_matrix = confusion_matrix(true_labels, predictions, labels=[0, 1, 2, 3])

    # Performance metrics
    throughput = total / duration if duration > 0 else 0.0
    avg_latency = duration / total if total > 0 else 0.0

    return {
        'accuracy': accuracy,
        'top2_accuracy': top2_accuracy,
        'correct': correct,
        'total': total,
        'ece': ece,
        'confidences': confidences,
        'predictions': predictions,
        'true_labels': true_labels,
        'confusion_matrix': conf_matrix,
        'throughput': throughput,
        'avg_latency': avg_latency,
    }

import time
import gc

def compute_model_flops(model, seq_length=512):
    """
    Estimate FLOPs per forward pass for transformer model.

    Formula: FLOPs ≈ 2 * active_params * seq_length
    For MoE: multiply by sparsity factor (active_experts / total_experts)
    """
    config = model.config
    n_layers = config.num_hidden_layers
    d_model = config.hidden_size
    intermediate_size = config.intermediate_size
    vocab_size = config.vocab_size

    # Attention FLOPs per layer: 4 * seq_len * d_model^2
    attention_flops = 4 * seq_length * d_model * d_model * n_layers

    # FFN FLOPs per layer: 2 * seq_len * d_model * intermediate_size * 2
    ffn_flops = 8 * seq_length * d_model * intermediate_size * n_layers

    total_flops = attention_flops + ffn_flops

    # Check if MoE model
    is_moe = False
    sparsity_factor = 1.0
    
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                num_experts_per_tok = layer.mlp.num_experts_per_tok
                sparsity_factor = num_experts_per_tok / num_experts
                break
    except:
        pass

    if is_moe:
        # Apply sparsity to FFN only (attention is unchanged)
        total_flops = attention_flops + (ffn_flops * sparsity_factor)

    return total_flops


def compute_throughput_metrics(model, tokenizer, eval_dataset, max_samples=100):
    """
    Measure throughput and latency metrics.

    Returns:
        - tokens_per_second: Throughput in tokens/sec
        - ms_per_token: Latency in ms/token
        - samples_per_second: Sample throughput
        - total_time: Total evaluation time
    """
    model.eval()

    # Take subset
    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    total_tokens = 0
    sample_times = []

    with torch.no_grad():
        for example in samples:
            prompt = example['formatted_prompt']

            # Tokenize
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                             max_length=MAX_LENGTH).to("cuda")
            num_tokens = inputs['input_ids'].shape[1]
            total_tokens += num_tokens

            # Time inference
            start = time.time()
            _ = model(**inputs)
            sample_time = time.time() - start
            sample_times.append(sample_time)

    total_time = sum(sample_times)
    tokens_per_second = total_tokens / total_time if total_time > 0 else 0
    ms_per_token = (total_time / total_tokens * 1000) if total_tokens > 0 else 0
    samples_per_second = len(samples) / total_time if total_time > 0 else 0

    return {
        'tokens_per_second': tokens_per_second,
        'ms_per_token': ms_per_token,
        'samples_per_second': samples_per_second,
        'total_time': total_time,
    }


def compute_parameter_efficiency(model, num_experts_per_tok=1):
    """
    Calculate parameter efficiency metrics.

    Returns:
        - total_params: Total model parameters
        - active_params: Parameters used per forward pass
        - trainable_params: Parameters being trained
        - sparsity_ratio: Fraction of params used (active/total)
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Check if MoE
    is_moe = False
    try:
        base_model = model
        if hasattr(model, 'base_model'):
            base_model = model.base_model
        if hasattr(base_model, 'model'):
            base_model = base_model.model
        
        layers = base_model.layers if hasattr(base_model, 'layers') else base_model.model.layers
        
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                is_moe = True
                num_experts = layer.mlp.num_experts
                break
    except:
        pass

    if is_moe:
        # Calculate active params correctly for MoE
        # Active = attention params + (k/n * expert params)
        # where k = experts per token, n = total experts
        
        # Get total expert parameters across all layers
        total_expert_params = 0
        for layer in layers:
            if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'num_experts'):
                # Sum all expert FFN parameters in this layer
                if hasattr(layer.mlp, 'gate_proj') and isinstance(layer.mlp.gate_proj, list):
                    for expert_idx in range(num_experts):
                        total_expert_params += sum(p.numel() for p in layer.mlp.gate_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.up_proj[expert_idx].parameters())
                        total_expert_params += sum(p.numel() for p in layer.mlp.down_proj[expert_idx].parameters())
        
        # Active expert params = (k/n) * total expert params
        sparsity = num_experts_per_tok / num_experts
        active_expert_params = int(total_expert_params * sparsity)
        
        # Non-expert params (attention, embeddings, etc)
        non_expert_params = total_params - total_expert_params
        
        # Total active = all non-expert params + active expert params
        active_params = non_expert_params + active_expert_params
    else:
        active_params = total_params

    sparsity_ratio = active_params / total_params if total_params > 0 else 1.0

    return {
        'total_params': total_params,
        'active_params': active_params,
        'trainable_params': trainable_params,
        'sparsity_ratio': sparsity_ratio,
    }


def compute_memory_metrics(model):
    """
    Compute memory usage statistics.

    Returns:
        - model_size_mb: Model size in memory
        - gpu_memory_allocated_gb: GPU memory allocated
        - gpu_memory_reserved_gb: GPU memory reserved
    """
    # Model size
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    model_size_mb = (param_size + buffer_size) / 1024 / 1024

    # GPU memory
    if torch.cuda.is_available():
        gpu_memory_allocated_gb = torch.cuda.memory_allocated() / 1024 / 1024 / 1024
        gpu_memory_reserved_gb = torch.cuda.memory_reserved() / 1024 / 1024 / 1024
    else:
        gpu_memory_allocated_gb = 0
        gpu_memory_reserved_gb = 0

    return {
        'model_size_mb': model_size_mb,
        'gpu_memory_allocated_gb': gpu_memory_allocated_gb,
        'gpu_memory_reserved_gb': gpu_memory_reserved_gb,
    }

print("evaluation functions defined")

evaluation functions defined


In [13]:
def collect_router_statistics(model, eval_dataset, tokenizer, answer_tokens,
                               max_samples=500, device="cuda"):
    """
    Collect router statistics from MoE model during inference.

    Args:
        model: MoE model
        eval_dataset: Dataset to evaluate on
        tokenizer: Tokenizer
        answer_tokens: Answer token IDs
        max_samples: Number of samples to analyze
        device: Device to run on

    Returns:
        Dict with router statistics
    """
    model.eval()

    # Detect MoE config
    first_moe_layer = None
    for layer in model.model.layers:
        if hasattr(layer.mlp, 'num_experts'):
            first_moe_layer = layer.mlp
            break

    if first_moe_layer:
        num_experts = first_moe_layer.num_experts
        num_experts_per_tok = first_moe_layer.num_experts_per_tok
    else:
        num_experts = globals().get('NUM_EXPERTS', 8)
        num_experts_per_tok = globals().get('NUM_EXPERTS_PER_TOK', 2)

    # Enable router logits collection
    for layer in model.model.layers:
        if hasattr(layer.mlp, 'forward'):
            layer.mlp._collect_router_logits = True

    # Storage for router statistics
    router_stats = {
        'expert_selections': defaultdict(lambda: np.zeros(num_experts)),
        'expert_confidence': defaultdict(list),
        'per_layer_selection': [np.zeros(num_experts) for _ in range(len(model.model.layers))],
        'per_subject_routing': defaultdict(lambda: defaultdict(lambda: np.zeros(num_experts))),
    }

    samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    print(f"Collecting router statistics from {len(samples)} samples (Experts: {num_experts}, k: {num_experts_per_tok})...")

    with torch.no_grad():
        for example in tqdm(samples, desc="Collecting router stats"):
            prompt = example['formatted_prompt']
            subject = example.get('subject', 'default')

            # Tokenize
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            # Forward pass
            outputs = model(**inputs)

            # Collect router information from each layer
            for layer_idx, layer in enumerate(model.model.layers):
                if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
                    router_probs = layer.mlp._last_router_probs.cpu().numpy()

                    # Average across all tokens in sequence
                    avg_probs = router_probs.mean(axis=0)

                    # Track which experts are selected (top-k)
                    top_experts = np.argsort(avg_probs)[-num_experts_per_tok:]

                    # Update statistics
                    router_stats['per_layer_selection'][layer_idx] += avg_probs

                    for expert_idx in top_experts:
                        router_stats['expert_selections']['overall'][expert_idx] += 1
                        router_stats['per_subject_routing'][subject][layer_idx][expert_idx] += 1

                    # Track confidence (probability of top expert)
                    router_stats['expert_confidence'][layer_idx].append(avg_probs.max())

    # Normalize statistics
    for layer_idx in range(len(router_stats['per_layer_selection'])):
        total = router_stats['per_layer_selection'][layer_idx].sum()
        if total > 0:
            router_stats['per_layer_selection'][layer_idx] /= total

    # Disable router collection
    for layer in model.model.layers:
        if hasattr(layer.mlp, '_collect_router_logits'):
            layer.mlp._collect_router_logits = False

    print("Router statistics collected!")
    return router_stats


def visualize_router_statistics(router_stats, title="MoE Router Analysis"):
    """
    Create comprehensive visualizations of router behavior.

    Args:
        router_stats: Router statistics from collect_router_statistics
        title: Title for the analysis
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Dynamically detect num_experts from data
    expert_usage = router_stats['expert_selections']['overall']
    num_experts = len(expert_usage)

    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # 1. Expert utilization across all layers
    ax1 = fig.add_subplot(gs[0, :2])
    if expert_usage.sum() > 0:
        expert_usage_norm = expert_usage / expert_usage.sum()
    else:
        expert_usage_norm = expert_usage

    bars = ax1.bar(range(num_experts), expert_usage_norm, color='steelblue', alpha=0.7)
    ax1.axhline(1/num_experts, color='red', linestyle='--', label='Uniform distribution')
    ax1.set_xlabel('Expert ID', fontsize=12)
    ax1.set_ylabel('Selection Frequency', fontsize=12)
    ax1.set_title('Overall Expert Utilization (All Layers)', fontsize=14, fontweight='bold')
    ax1.set_xticks(range(num_experts))
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Add percentage labels on bars
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height*100:.1f}%', ha='center', va='bottom', fontsize=9)

    # 2. Expert utilization heatmap across layers
    ax2 = fig.add_subplot(gs[0, 2])
    layer_expert_matrix = np.array(router_stats['per_layer_selection'])
    sns.heatmap(layer_expert_matrix.T, cmap='YlOrRd', ax=ax2, cbar_kws={'label': 'Selection Prob'})
    ax2.set_xlabel('Layer', fontsize=10)
    ax2.set_ylabel('Expert ID', fontsize=10)
    ax2.set_title('Expert Selection Heatmap\n(Layer vs Expert)', fontsize=12, fontweight='bold')

    # 3. Load balance score per layer
    ax3 = fig.add_subplot(gs[1, 0])
    load_balance_scores = []
    for layer_probs in router_stats['per_layer_selection']:
        # Compute entropy as measure of load balance
        if layer_probs.sum() > 0:
            layer_probs_norm = layer_probs / layer_probs.sum()
            entropy = -np.sum(layer_probs_norm * np.log(layer_probs_norm + 1e-10))
            normalized_entropy = entropy / np.log(num_experts)
        else:
            normalized_entropy = 0
        load_balance_scores.append(normalized_entropy)

    ax3.plot(load_balance_scores, marker='o', linewidth=2, markersize=4)
    ax3.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect balance')
    ax3.axhline(0.5, color='orange', linestyle='--', alpha=0.5, label='50% balance')
    ax3.set_xlabel('Layer', fontsize=10)
    ax3.set_ylabel('Load Balance Score', fontsize=10)
    ax3.set_title('Load Balancing Across Layers\n(1.0 = perfect balance)', fontsize=12, fontweight='bold')
    ax3.legend()
    ax3.grid(alpha=0.3)
    ax3.set_ylim([0, 1.1])

    # 4. Router confidence distribution
    ax4 = fig.add_subplot(gs[1, 1])
    all_confidences = []
    for layer_confs in router_stats['expert_confidence'].values():
        all_confidences.extend(layer_confs)

    ax4.hist(all_confidences, bins=50, color='purple', alpha=0.7, edgecolor='black')
    ax4.axvline(np.mean(all_confidences), color='red', linestyle='--',
                linewidth=2, label=f"Mean: {np.mean(all_confidences):.3f}")
    ax4.set_xlabel('Router Confidence (Max Prob)', fontsize=10)
    ax4.set_ylabel('Frequency', fontsize=10)
    ax4.set_title('Distribution of Router Confidence', fontsize=12, fontweight='bold')
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)

    # 5. Expert specialization: variance across layers
    ax5 = fig.add_subplot(gs[1, 2])
    expert_variances = []
    for expert_id in range(num_experts):
        expert_across_layers = layer_expert_matrix[:, expert_id]
        variance = np.var(expert_across_layers)
        expert_variances.append(variance)

    ax5.bar(range(num_experts), expert_variances, color='teal', alpha=0.7)
    ax5.set_xlabel('Expert ID', fontsize=10)
    ax5.set_ylabel('Variance Across Layers', fontsize=10)
    ax5.set_title('Expert Specialization\n(Higher = more layer-specific)', fontsize=12, fontweight='bold')
    ax5.set_xticks(range(num_experts))
    ax5.grid(axis='y', alpha=0.3)

    # 6. Per-layer confidence box plots
    ax6 = fig.add_subplot(gs[2, :])
    layer_confidence_data = [router_stats['expert_confidence'][i]
                             for i in range(len(router_stats['expert_confidence']))]
    bp = ax6.boxplot(layer_confidence_data, patch_artist=True, showmeans=True)

    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)

    ax6.set_xlabel('Layer', fontsize=12)
    ax6.set_ylabel('Router Confidence', fontsize=12)
    ax6.set_title('Router Confidence Distribution Across Layers', fontsize=14, fontweight='bold')
    ax6.grid(axis='y', alpha=0.3)

    plt.suptitle(title, fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('router_visualization.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to 'router_visualization.png'")
    plt.show()

    # Print statistics
  
    print("ROUTER STATISTICS SUMMARY")
 

    print(f"Average load balance score: {np.mean(load_balance_scores):.4f}")
    print(f"Average router confidence: {np.mean(all_confidences):.4f}")
    print(f"Std dev of expert usage: {np.std(expert_usage_norm):.4f}")

    # Check for router collapse
    max_expert_usage = expert_usage_norm.max()
    if max_expert_usage > 0.3:
        print(f"\nWARNING: Potential router collapse detected!")
        print(f"   Expert {expert_usage_norm.argmax()} is selected {max_expert_usage*100:.1f}% of the time")
    else:
        print(f"\nNo router collapse detected (max usage: {max_expert_usage*100:.1f}%)")


In [14]:
from transformers import TrainerCallback
from torch.utils.data import DataLoader
import gc

class ComprehensiveEvalCallback(TrainerCallback):
    """
    Custom callback that evaluates and logs all Priority 1-3 metrics every N steps.
    """

    def __init__(self, eval_dataset_for_accuracy, eval_tokenized_for_loss,
                 tokenizer, answer_tokens, eval_steps=50,
                 accuracy_samples=200, device="cuda"):
        super().__init__()
        self.eval_dataset_for_accuracy = eval_dataset_for_accuracy
        self.eval_tokenized_for_loss = eval_tokenized_for_loss
        self.tokenizer = tokenizer
        self.answer_tokens = answer_tokens
        self.eval_steps = eval_steps
        self.accuracy_samples = accuracy_samples
        self.device = device

        self.metrics_history = []
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

        print(f"Evaluating every {self.eval_steps} steps")

        return control

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.eval_steps == 0 and state.global_step > 0:
            self._evaluate_and_log(args, state, model)
        return control

    def on_epoch_end(self, args, state, control, model=None, **kwargs):

        print(f"END OF EPOCH {state.epoch:.0f}")

        self._evaluate_and_log(args, state, model, is_epoch_end=True)
        return control

    def _evaluate_and_log(self, args, state, model, is_epoch_end=False):
        """Compute and log all metrics."""

        # Get training loss from state
        train_loss = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'loss' in log:
                    train_loss = log['loss']
                    break

        # Get learning rate
        learning_rate = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'learning_rate' in log:
                    learning_rate = log['learning_rate']
                    break

        # Compute evaluation loss and throughput
        eval_loss, eval_throughput, eval_latency = self._compute_eval_loss(model)

        # Compute perplexity
        perplexity = torch.exp(torch.tensor(eval_loss)).item()

        # Comprehensive MMLU evaluation
        mmlu_results = evaluate_mmlu_comprehensive(
            model=model,
            tokenizer=self.tokenizer,
            eval_dataset=self.eval_dataset_for_accuracy,
            answer_tokens=self.answer_tokens,
            device=self.device,
            max_samples=self.accuracy_samples,
            show_progress=False
        )

        # GPU memory
        peak_gpu_memory = self._get_peak_gpu_memory()

        # Calculate elapsed time
        elapsed = time.time() - self.start_time

        # Gradient norm (if available)
        grad_norm = None
        if len(state.log_history) > 0:
            for log in reversed(state.log_history):
                if 'grad_norm' in log:
                    grad_norm = log['grad_norm']
                    break

        # Store metrics
        metrics = {
            'step': state.global_step,
            'epoch': state.epoch,
            'train_loss': train_loss,
            'eval_loss': eval_loss,
            'perplexity': perplexity,
            'learning_rate': learning_rate,
            'grad_norm': grad_norm,
            # MMLU metrics
            'mmlu_accuracy': mmlu_results['accuracy'],
            'mmlu_top2_accuracy': mmlu_results['top2_accuracy'],
            'mmlu_ece': mmlu_results['ece'],
            'mmlu_correct': mmlu_results['correct'],
            'mmlu_total': mmlu_results['total'],
            # Performance metrics
            'throughput': mmlu_results['throughput'],
            'avg_latency': mmlu_results['avg_latency'],
            'eval_throughput': eval_throughput,
            'eval_latency': eval_latency,
            'peak_gpu_memory_gb': peak_gpu_memory,
            'elapsed_time': elapsed,
        }
        self.metrics_history.append(metrics)

        # Log to wandb
        wandb.log({
            'step': state.global_step,
            'epoch': state.epoch,
            'train/loss': train_loss,
            'eval/loss': eval_loss,
            'eval/perplexity': perplexity,
            'train/learning_rate': learning_rate,
            'train/grad_norm': grad_norm,
            'mmlu/accuracy': mmlu_results['accuracy'],
            'mmlu/top2_accuracy': mmlu_results['top2_accuracy'],
            'mmlu/ece': mmlu_results['ece'],
            'performance/throughput': mmlu_results['throughput'],
            'performance/latency': mmlu_results['avg_latency'],
            'system/peak_gpu_memory_gb': peak_gpu_memory,
        })

        # Print metrics
        step_info = f"Epoch {state.epoch:.2f}" if is_epoch_end else f"Step {state.global_step}"

        print(f"EVALUATION at {step_info}")
        print(f"  Train Loss:      {train_loss:.4f}" if train_loss else "  Train Loss:      N/A")
        print(f"  Eval Loss:       {eval_loss:.4f}")
        print(f"  MMLU Accuracy:   {mmlu_results['accuracy']:.4f} ({mmlu_results['correct']}/{mmlu_results['total']})")
        print(f"  Perplexity:      {perplexity:.2f}")
        print(f"  Throughput:      {mmlu_results['throughput']:.2f} samples/sec")
        print(f"  Avg Latency:     {mmlu_results['avg_latency']:.4f} sec/sample")
        print(f"  Peak GPU Memory: {peak_gpu_memory:.2f} GB")
        print(f"  ECE (Calibration): {mmlu_results['ece']:.4f}")
        print(f"  Top-2 Accuracy:  {mmlu_results['top2_accuracy']:.4f}")
        if learning_rate:
            print(f"  Learning Rate:   {learning_rate:.2e}")
        if grad_norm:
            print(f"  Gradient Norm:   {grad_norm:.4f}")

        print(f" Elapsed Time:    {elapsed/60:.1f} min")


        # Clean up
        torch.cuda.empty_cache()
        gc.collect()
        model.train()

    def _compute_eval_loss(self, model, num_batches=50):
        """Compute average loss on evaluation set."""
        model.eval()
        total_loss = 0
        num_samples = 0

        eval_dataloader = DataLoader(
            self.eval_tokenized_for_loss,
            batch_size=1,
            shuffle=False
        )

        start_time = time.time()

        with torch.no_grad():
            for i, batch in enumerate(eval_dataloader):
                if i >= num_batches:
                    break

                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = model(**batch)
                total_loss += outputs.loss.item()
                num_samples += 1

        end_time = time.time()
        duration = end_time - start_time

        avg_loss = total_loss / num_samples if num_samples > 0 else 0
        throughput = num_samples / duration if duration > 0 else 0.0
        latency = duration / num_samples if num_samples > 0 else 0.0

        return avg_loss, throughput, latency

    def _get_peak_gpu_memory(self):
        """Get peak GPU memory and reset stats."""
        if torch.cuda.is_available():
            max_mem = torch.cuda.max_memory_allocated() / (1024**3)
            torch.cuda.reset_peak_memory_stats()
            return max_mem
        return 0.0

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time

        print(f"TRAINING COMPLETED in {elapsed/60:.1f} minutes")

        # Print summary table
        if self.metrics_history:
            print("TRAINING METRICS SUMMARY:")
            print(f"{'Step':<8} {'Epoch':<7} {'Train':<8} {'Eval':<8} {'Perp':<7} {'MMLU':<8} {'Top-2':<8} {'ECE':<7} {'Latency':<9}")
            print(f"{'':8} {'':7} {'Loss':<8} {'Loss':<8} {'':7} {'Acc':<8} {'Acc':<8} {'':7} {'(sec)':<9}")
            print("" * 80)
            for m in self.metrics_history:
                train_loss_str = f"{m['train_loss']:.4f}" if m['train_loss'] else "N/A"
                print(f"{m['step']:<8} {m['epoch']:<7.2f} {train_loss_str:<8} {m['eval_loss']:<8.4f} "
                      f"{m['perplexity']:<7.2f} {m['mmlu_accuracy']:<8.4f} {m['mmlu_top2_accuracy']:<8.4f} "
                      f"{m['mmlu_ece']:<7.4f} {m['avg_latency']:<9.4f}")

        return control


def evaluate_mmlu_batched(model, tokenizer, eval_dataset, answer_tokens, device="cuda",
                          batch_size=8, max_samples=None):
    """
    Batched MMLU evaluation for faster inference.
    """
    model.eval()
    samples = eval_dataset
    if max_samples is not None:
        samples = eval_dataset.select(range(min(max_samples, len(eval_dataset))))

    answer_token_ids = torch.tensor([answer_tokens[c] for c in ["A","B","C","D"]]).to(device)
    correct = top2_correct = 0
    total = 0

    def collate(batch):
        prompts = [ex["formatted_prompt"] for ex in batch]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        return inputs, batch

    loader = DataLoader(samples, batch_size=batch_size, shuffle=False, collate_fn=collate)

    with torch.no_grad():
        for inputs, batch in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            last_logits = outputs.logits[:, -1, :]                 # [B, V]
            answer_logits = last_logits[:, answer_token_ids]       # [B, 4]
            probs = torch.softmax(answer_logits, dim=-1)           # [B, 4]

            pred_idx = probs.argmax(dim=-1)
            top2 = probs.topk(2, dim=-1).indices

            for i, ex in enumerate(batch):
                true_idx = answer_to_idx[ex["answer"]]
                total += 1
                correct += int(pred_idx[i].item() == true_idx)
                top2_correct += int(true_idx in top2[i])

    accuracy = correct / total if total else 0.0
    top2_accuracy = top2_correct / total if total else 0.0
    return {"accuracy": accuracy, "top2_accuracy": top2_accuracy, "correct": correct, "total": total}


print(" Comprehensive evaluation callback with batched eval defined")


 Comprehensive evaluation callback with batched eval defined


# Dense Model Baseline Evaluation

In [15]:

print("BASELINE EVALUATION")


# 1. MMLU Accuracy
baseline_results = evaluate_mmlu_comprehensive(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs estimation
print("\nComputing FLOPs...")
baseline_flops = compute_model_flops(base_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print("Measuring throughput...")
baseline_throughput = compute_throughput_metrics(
    model=base_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print("Analyzing parameter efficiency...")
baseline_params = compute_parameter_efficiency(
    model=base_model,
    num_experts_per_tok=1  # Dense model
)

# 5. Memory metrics
print("Collecting memory metrics...")
baseline_memory = compute_memory_metrics(base_model)

# Combine all metrics
baseline_comprehensive = {
    **baseline_results,
    'flops': baseline_flops,
    **baseline_throughput,
    **baseline_params,
    **baseline_memory,
}

# Display comprehensive results

print("COMPREHENSIVE BASELINE METRICS")


print("Accuracy Metrics:")
print(f"  MMLU Accuracy: {baseline_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {baseline_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {baseline_comprehensive['ece']:.4f}")

print("\n Computational Efficiency:")
print(f"  FLOPs per forward pass: {baseline_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {baseline_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {baseline_comprehensive['ms_per_token']:.2f}")
print(f"  Samples/second: {baseline_comprehensive['samples_per_second']:.2f}")

print("\n Parameter Efficiency:")
print(f"  Total parameters: {baseline_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {baseline_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {baseline_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {baseline_comprehensive['sparsity_ratio']:.2%}")

print("\n Memory Usage:")
print(f"  Model size: {baseline_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
print(f"  GPU reserved: {baseline_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# Save comprehensive results
import json
import os
os.makedirs("results", exist_ok=True)
with open("results/baseline_comprehensive.json", 'w') as f:
    json.dump({k: v for k, v in baseline_comprehensive.items()
               if not isinstance(v, np.ndarray)}, f, indent=2, default=str)

# Log to wandb with detailed metrics
try:
    if wandb.run is not None:
        wandb.log({
            'baseline/accuracy': baseline_comprehensive['accuracy'],
            'baseline/flops_billions': baseline_comprehensive['flops']/1e9,
            'baseline/tokens_per_second': baseline_comprehensive['tokens_per_second'],
            'baseline/ms_per_token': baseline_comprehensive['ms_per_token'],
            'baseline/active_params_billions': baseline_comprehensive['active_params']/1e9,
            'baseline/gpu_memory_gb': baseline_comprehensive['gpu_memory_allocated_gb'],
        })
except:
    pass

# Store for later comparison
baseline_accuracy = baseline_comprehensive['accuracy']

print("\n Comprehensive baseline evaluation complete!")

BASELINE EVALUATION


Evaluating: 100%|██████████| 1000/1000 [01:31<00:00, 10.89it/s]



Computing FLOPs...
Measuring throughput...
Analyzing parameter efficiency...
COMPREHENSIVE BASELINE METRICS
Accuracy Metrics:
  MMLU Accuracy: 0.6590
  Top-2 Accuracy: 0.8370
  ECE: 0.0796

 Computational Efficiency:
  FLOPs per forward pass: 8796.09G
  Tokens/second: 3121.24
  ms/token: 0.32
  Samples/second: 10.97

 Parameter Efficiency:
  Total parameters: 7.24B
  Active parameters: 7.24B
  Trainable parameters: 262.41M
  Sparsity ratio: 100.00%

 Memory Usage:
  Model size: 7156.51 MB
  GPU allocated: 7.03 GB
  GPU reserved: 7.86 GB

 Comprehensive baseline evaluation complete!


# Knowledge distillation setup

In [16]:
# Teacher = dense baseline
teacher_model = base_model.eval()
teacher_baseline_results = baseline_comprehensive.copy()
print(f" Teacher model set (dense). Teacher accuracy: {teacher_baseline_results['accuracy']:.4f}")

# KD config 1: Output-only distillation (stable)
KD_CONFIG_STANDARD = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': False,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.001,
    'name': 'Standard KD',
}

# KD config 2: Output + light router constraints (MoE-stable)
KD_CONFIG_ROUTER_STABLE = {
    'kd_alpha': 0.6,
    'temperature': 5.0,
    'routing_kd_weight': 0.1,
    'expert_spec_weight': 0.0,
    'enable_routing_kd': True,
    'enable_ka': False,
    'enable_sar': False,
    'enable_non_activated': False,
    'router_aux_loss_coef': 0.01,
    'name': 'Router-Stable KD',
}

print("KD configs ready (Standard, Router-Stable)")

 Teacher model set (dense). Teacher accuracy: 0.6590
KD configs ready (Standard, Router-Stable)


# Baseline MoE implementation

In [17]:
import torch
import torch.nn as nn
from typing import Optional

class MoELayer(nn.Module):
    """
     MoE Layer with:
    - Proper router initialization for top-2 routing
    - VECTORIZED output combination (FAST - no Python loops)
    - No expert noise - all experts start identical
    """
    
    def __init__(self, hidden_size, intermediate_size, num_experts=8,
                 num_experts_per_tok=2, router_jitter_noise=0.0,
                 router_aux_loss_coef=0.001, bnb_config=None, device="cuda",
                 init_on_cpu=True, dtype=torch.bfloat16, enable_cpu_offload=True):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.router_jitter_noise = router_jitter_noise
        self.router_aux_loss_coef = router_aux_loss_coef
        self.bnb_config = bnb_config
        self.device = device
        self.dtype = dtype
        
        if bnb_config is None and enable_cpu_offload:
            enable_cpu_offload = False
        self.enable_cpu_offload = enable_cpu_offload
        
        # Configure linear layer class
        if bnb_config is not None:
            from bitsandbytes.nn import Linear4bit
            LinearClass = Linear4bit
            self.compute_dtype = torch.bfloat16
            linear_kwargs = {
                'device': device,
                'compute_dtype': bnb_config.bnb_4bit_compute_dtype if hasattr(bnb_config, 'bnb_4bit_compute_dtype') else torch.bfloat16,
                'quant_type': bnb_config.bnb_4bit_quant_type if hasattr(bnb_config, 'bnb_4bit_quant_type') else 'nf4',
                'compress_statistics': bnb_config.bnb_4bit_use_double_quant if hasattr(bnb_config, 'bnb_4bit_use_double_quant') else False,
            }
            init_device = device
            self.enable_cpu_offload = False
        else:
            LinearClass = nn.Linear
            self.compute_dtype = dtype
            init_device = 'cpu' if (init_on_cpu or enable_cpu_offload) else device
            linear_kwargs = {'device': init_device, 'dtype': dtype}
        
        router_dtype = torch.bfloat16 if bnb_config is not None else dtype
        
        # Initialize experts
        self.gate_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.up_proj = nn.ModuleList([
            LinearClass(hidden_size, intermediate_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        self.down_proj = nn.ModuleList([
            LinearClass(intermediate_size, hidden_size, bias=False, **linear_kwargs)
            for _ in range(num_experts)
        ])
        
        # Router initialization for top-2 routing
        self.router = nn.Linear(hidden_size, num_experts, bias=False, device=device, dtype=router_dtype)
        with torch.no_grad():
            self.router.weight.zero_()
            expert0_bias = 3.0 if bnb_config is not None else 2.0
            self.router.weight[0, :] = expert0_bias
            expert1_bias = 2.5 if bnb_config is not None else 1.5
            self.router.weight[1, :] = expert1_bias
            if num_experts > 2:
                negative_bias = -0.5 if bnb_config is not None else -0.3
                self.router.weight[2:, :] = negative_bias
                noise_scale = 0.01 if bnb_config is not None else 0.02
                self.router.weight[2:, :] += torch.randn(
                    num_experts-2, hidden_size, device=device, dtype=router_dtype
                ) * noise_scale
        
        self._last_router_probs = None
        self._collect_router_logits = False
        self._experts_on_gpu = set()
    
    def forward(self, hidden_states):
        """
        VECTORIZED forward pass (FAST - no Python loops).
        """
        original_dtype = hidden_states.dtype
        hidden_states_reshaped = hidden_states.view(-1, hidden_states.size(-1))  # [B*S, H]
        
        # Router computation
        router_input = hidden_states_reshaped.to(self.router.weight.dtype)
        router_logits = self.router(router_input)  # [B*S, num_experts]
        
        if self.training and self.router_jitter_noise > 0:
            router_logits = router_logits + torch.normal(
                0, self.router_jitter_noise, size=router_logits.shape, device=router_logits.device
            )
        
        router_probs = torch.softmax(router_logits, dim=-1)  # [B*S, num_experts]
        top_k_probs, top_k_indices = torch.topk(router_probs, self.num_experts_per_tok, dim=-1)  # [B*S, k]

        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        if self.training:
            self._last_router_probs = router_probs.view(hidden_states.size(0), hidden_states.size(1), -1)
        
        # VECTORIZED output combination (FAST)
        output = torch.zeros_like(hidden_states_reshaped)
        
        for expert_idx in range(self.num_experts):
            expert_mask = (top_k_indices == expert_idx).any(dim=-1)  # [B*S]
            if expert_mask.sum() == 0:
                continue
            
            # VECTORIZED: Compute weights for all tokens at once
            expert_selected_mask = (top_k_indices == expert_idx)  # [B*S, k]
            expert_weights = (top_k_probs * expert_selected_mask.float()).sum(dim=-1)  # [B*S]
            
            # Apply expert FFN
            expert_hidden = hidden_states_reshaped[expert_mask]
            
            if self.compute_dtype is not None and expert_hidden.dtype != self.compute_dtype:
                expert_hidden = expert_hidden.to(dtype=self.compute_dtype)
            
            gate_out = torch.nn.functional.silu(self.gate_proj[expert_idx](expert_hidden))
            up_out = self.up_proj[expert_idx](expert_hidden)
            expert_out = gate_out * up_out
            expert_out = self.down_proj[expert_idx](expert_out)
            expert_out = expert_out.to(dtype=output.dtype)
            
            # VECTORIZED: Weighted combination
            output[expert_mask] += expert_weights[expert_mask].unsqueeze(-1) * expert_out
        
        output = output.view_as(hidden_states)
        return output.to(original_dtype)
    
    def compute_auxiliary_loss(self):
        if self._last_router_probs is None:
            return torch.tensor(0.0, device=self.router.weight.device)
        expert_freq = self._last_router_probs.mean(dim=[0, 1])
        router_confidence = self._last_router_probs.mean(dim=[0, 1])
        aux_loss = torch.sum(expert_freq * router_confidence) * self.num_experts
        return aux_loss * self.router_aux_loss_coef

print("MoELayer class defined")
print("Router initialized for top-2 routing")

MoELayer class defined
Router initialized for top-2 routing


In [18]:
# ============================================================================
# FIXED replace_ffn_with_moe - No Double Quantization, No Expert Noise
# ============================================================================
# Fixes:
# 1.  No expert noise - all experts start identical (preserves pretrained knowledge)
# 2.  Minimized quantization errors (proper dequantization)
# 3.  Proper weight extraction and copying
# ============================================================================

def replace_ffn_with_moe(model, num_experts=8, num_experts_per_tok=2,
                         router_jitter_noise=0.0, router_aux_loss_coef=0.001,
                         bnb_config=None, ram_threshold=50.0, use_disk_offload=True,
                         layer_indices=None, half_width=False, enable_cpu_offload=True):
    """
    FIXED: Replace dense FFN layers with MoE layers.
    
    Key improvements:
    - No expert noise - all experts start identical (preserves pretrained knowledge)
    - Minimized quantization errors (Linear4bit handles quantization automatically)
    - Proper weight extraction and copying
    """
    import gc
    import psutil
    import os
    from bitsandbytes.nn import Linear4bit as BnbLinear
    from bitsandbytes.functional import dequantize_4bit
    
    # Set CUDA allocator config
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
    
    config = model.config
    hidden_size = config.hidden_size
    
    # Use half-width experts if specified
    if half_width:
        intermediate_size = config.intermediate_size // 2
        print(f"   Using HALF-WIDTH experts (intermediate_size // 2)")
    else:
        intermediate_size = config.intermediate_size
    
    print(f"Model configuration:")
    print(f"  Hidden size: {hidden_size}")
    print(f"  Original intermediate size: {config.intermediate_size}")
    print(f"  MoE intermediate size: {intermediate_size}")
    print(f"  Number of layers: {config.num_hidden_layers}")
    print(f"  Experts per layer: {num_experts}")
    print(f"  Experts per token: {num_experts_per_tok}")
    
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available!")
    
    device = torch.device("cuda")
    if bnb_config and hasattr(bnb_config, 'bnb_4bit_compute_dtype'):
        compute_dtype = bnb_config.bnb_4bit_compute_dtype
    elif bnb_config and hasattr(bnb_config, 'llm_int8_threshold'):
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.bfloat16
    
    print(f"\n Using GPU for weight processing")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Compute dtype: {compute_dtype}")
    
    # Helper functions
    def check_ram():
        return psutil.virtual_memory().percent
    
    def cleanup(aggressive=False):
        gc.collect()
        torch.cuda.empty_cache()
        if aggressive:
            torch.cuda.synchronize()
        try:
            import ctypes
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except:
            pass
        gc.collect()
    
    def print_memory_stats(label=""):
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"    [{label}] GPU: {allocated:.2f}GB alloc / {reserved:.2f}GB reserved")
    
    def extract_weight(linear_layer, expected_shape, keep_on_cpu=True):
        """
        Extract and dequantize weight from a layer.
        FIXED: Proper dequantization for Linear4bit layers.
        """
        expected_numel = torch.Size(expected_shape).numel() if isinstance(expected_shape, (tuple, list)) else expected_shape.numel()
        
        if isinstance(linear_layer, BnbLinear):
            # FIXED: Proper dequantization for Linear4bit
            try:
                if not hasattr(linear_layer.weight, 'quant_state'):
                    raise AttributeError("linear_layer.weight.quant_state not found")
                
                weight = dequantize_4bit(
                    linear_layer.weight.data,
                    linear_layer.weight.quant_state
                ).to(compute_dtype)
            except Exception as e:
                print(f"[WARNING] dequantize_4bit failed: {e}. Trying fallback...")
                try:
                    weight = linear_layer.weight.data.to(compute_dtype)
                    print(f"[WARNING] Using fallback weight extraction. Shape: {weight.shape}")
                except Exception as e2:
                    raise RuntimeError(f"Failed to extract weight from Linear4bit. dequantize_4bit error: {e}. Fallback error: {e2}")
            
            # Ensure proper shape
            if weight.shape != expected_shape:
                if weight.numel() == expected_numel:
                    weight = weight.reshape(expected_shape)
                elif len(weight.shape) == 2 and len(expected_shape) == 2:
                    if weight.shape[0] == expected_shape[1] and weight.shape[1] == expected_shape[0]:
                        weight = weight.t()
                    elif weight.numel() == expected_numel:
                        weight = weight.reshape(expected_shape)
                elif len(weight.shape) == 1 or (len(weight.shape) == 2 and weight.shape[1] == 1):
                    if weight.numel() == expected_numel:
                        weight = weight.reshape(expected_shape)
                    else:
                        transposed_numel = expected_shape[1] * expected_shape[0]
                        if weight.numel() == transposed_numel:
                            weight = weight.reshape(expected_shape[1], expected_shape[0]).t()
            
            return weight.cpu() if keep_on_cpu else weight.cuda()
        else:
            weight = linear_layer.weight.data.to(compute_dtype)
            if weight.shape != expected_shape and weight.numel() == expected_numel:
                weight = weight.reshape(expected_shape)
            return weight.cpu() if keep_on_cpu else weight
    
    # Determine layers to process
    total_layers = len(model.model.layers)
    if layer_indices is None:
        target_layers = list(range(total_layers))
    else:
        target_layers = layer_indices
    print(f"\n  Processing {len(target_layers)} layers: {target_layers[:5]}{'...' if len(target_layers) > 5 else ''}\n")
    
    for i in target_layers:
        layer = model.model.layers[i]
        original_mlp = layer.mlp
        
        ram = check_ram()
        print(f"  Layer {i+1}/{total_layers} (RAM: {ram:.1f}%):")
        
        # Step 1: Extract all weights to CPU (memory efficient)
        print(f"    Extracting weights to CPU...", end=" ", flush=True)
        with torch.no_grad():
            # Extract weights and keep on CPU
            gate_w_full = extract_weight(original_mlp.gate_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            up_w_full = extract_weight(original_mlp.up_proj, (config.intermediate_size, hidden_size), keep_on_cpu=True)
            down_w_full = extract_weight(original_mlp.down_proj, (hidden_size, config.intermediate_size), keep_on_cpu=True)
            
            # Slice to target intermediate size if needed
            gate_w = gate_w_full[:intermediate_size, :].clone()
            up_w = up_w_full[:intermediate_size, :].clone()
            down_w = down_w_full[:, :intermediate_size].clone()
            
            # Delete full weights IMMEDIATELY
            del gate_w_full, up_w_full, down_w_full
            gc.collect()
        print("Done")
        
        # Step 2: Delete original MLP to free GPU memory
        print(f"    Deleting original MLP...", end=" ", flush=True)
        del original_mlp
        layer.mlp = None
        cleanup(aggressive=True)
        print("Done")
        
        # Step 3: Create MoE layer
        print(f"    Creating MoE layer...", end=" ", flush=True)
        moe_layer = MoELayer(
            hidden_size=hidden_size,
            intermediate_size=intermediate_size,
            num_experts=num_experts,
            num_experts_per_tok=num_experts_per_tok,
            router_jitter_noise=router_jitter_noise,
            router_aux_loss_coef=router_aux_loss_coef,
            bnb_config=bnb_config,
            device='cpu' if bnb_config is None else device,
            init_on_cpu=(bnb_config is None),
            dtype=compute_dtype,
            enable_cpu_offload=enable_cpu_offload
        )
        print("Done")
        
        # Step 4: Copy weights to all experts (on CPU) - FIXED: NO NOISE
        print(f"    Copying weights to {num_experts} experts (NO NOISE)...", end=" ", flush=True)
        with torch.no_grad():
            # Validate and fix shapes before copying
            gate_target_shape = moe_layer.gate_proj[0].weight.shape
            up_target_shape = moe_layer.up_proj[0].weight.shape
            down_target_shape = moe_layer.down_proj[0].weight.shape
            
            # Fix gate_w shape if needed
            if gate_w.shape != gate_target_shape:
                if gate_w.numel() == gate_target_shape.numel():
                    gate_w = gate_w.reshape(gate_target_shape)
                else:
                    gate_w = gate_w[:gate_target_shape[0], :gate_target_shape[1]]
            
            # Fix up_w shape if needed
            if up_w.shape != up_target_shape:
                if up_w.numel() == up_target_shape.numel():
                    up_w = up_w.reshape(up_target_shape)
                else:
                    up_w = up_w[:up_target_shape[0], :up_target_shape[1]]
            
            # Fix down_w shape if needed
            if down_w.shape != down_target_shape:
                if down_w.numel() == down_target_shape.numel():
                    down_w = down_w.reshape(down_target_shape)
                elif len(down_w.shape) == 1 or (len(down_w.shape) == 2 and down_w.shape[1] == 1):
                    if down_w.numel() == down_target_shape.numel():
                        down_w = down_w.reshape(down_target_shape)
                    else:
                        transposed_shape = (down_target_shape[1], down_target_shape[0])
                        if down_w.numel() == torch.Size(transposed_shape).numel():
                            down_w = down_w.reshape(transposed_shape).t()
                        else:
                            raise RuntimeError(
                                f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                                f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}."
                            )
                elif down_w.shape[0] == down_target_shape[0] and down_w.shape[1] >= down_target_shape[1]:
                    down_w = down_w[:, :down_target_shape[1]]
                elif down_w.shape[1] == down_target_shape[1] and down_w.shape[0] >= down_target_shape[0]:
                    down_w = down_w[:down_target_shape[0], :]
                elif down_w.shape[0] == down_target_shape[1] and down_w.shape[1] == down_target_shape[0]:
                    down_w = down_w.t()
                else:
                    raise RuntimeError(
                        f"Cannot reshape down_w from {down_w.shape} to {down_target_shape}. "
                        f"down_w.numel()={down_w.numel()}, target.numel()={down_target_shape.numel()}."
                    )
            
            # ========================================================================
            # FIXED: Copy weights to ALL experts identically (NO NOISE)
            # ========================================================================
            # All experts start with identical pretrained weights
            # Training will naturally differentiate them through gradient updates
            for idx in range(num_experts):
                moe_layer.gate_proj[idx].weight.copy_(gate_w)
                moe_layer.up_proj[idx].weight.copy_(up_w)
                moe_layer.down_proj[idx].weight.copy_(down_w)
                
                # NO NOISE ADDED - experts are identical initially
                # This preserves pretrained knowledge perfectly
                # The router is initialized to prefer experts 0 & 1 for top-2 routing
                # Training will naturally differentiate all experts
        
        print("Done")
        
        # Delete extracted weights BEFORE moving to GPU
        del gate_w, up_w, down_w
        gc.collect()
        
        # Step 5: Move entire ModuleLists to GPU at once (faster than one-by-one)
        print(f"    Moving to GPU...", end=" ", flush=True)
        moe_layer.gate_proj = moe_layer.gate_proj.to(device)
        moe_layer.up_proj = moe_layer.up_proj.to(device)
        moe_layer.down_proj = moe_layer.down_proj.to(device)
        moe_layer.router = moe_layer.router.to(device)
        print("Done")
        
        # Step 6: Replace and cleanup
        layer.mlp = moe_layer
        cleanup(aggressive=True)
        
        # Progress update every 4 layers
        if (i + 1) % 4 == 0:
            print_memory_stats(f"Layer {i+1}")
    
    cleanup(aggressive=True)
    
    print(f"\n Successfully replaced {len(target_layers)} FFN layers with MoE")
    print(f"  Expert dispatch: Efficient sparse routing (top-{num_experts_per_tok})")
    print(f"  CPU offloading: {'ENABLED' if enable_cpu_offload else 'DISABLED'}")
    print(f"  Params per expert: ~{(intermediate_size * hidden_size * 3) / 1e6:.1f}M")
    print(f"  Active params per token: ~{(intermediate_size * hidden_size * 3 * num_experts_per_tok) / 1e6:.1f}M")
    print(f"   All experts start identical (no noise)")
    print(f"   Router initialized for top-2 routing (experts 0 & 1 preferred)")
    
    # Final stats
    gpu_final = torch.cuda.memory_allocated() / 1e9
    ram_final = check_ram()
    print(f"  Final: GPU {gpu_final:.2f}GB | RAM {ram_final:.1f}%")
    return model

def compute_moe_auxiliary_loss(model, router_aux_loss_coef=0.001):
    """
    Compute auxiliary load balancing loss from all MoE layers.
    L_aux = sum(D_i * P_i) where:
    - D_i = expert frequency (actual utilization rate)
    - P_i = router probability (router confidence)

    Args:
        model: MoE model
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        aux_loss: Auxiliary loss value
    """
    total_aux_loss = torch.tensor(0.0, device=next(model.parameters()).device)

    # Collect router information from all MoE layers
    for layer in model.model.layers:
        if hasattr(layer.mlp, '_last_router_probs') and layer.mlp._last_router_probs is not None:
            router_probs = layer.mlp._last_router_probs

            # Expert frequency: how often each expert is selected (average probability)
            expert_freq = router_probs.mean(dim=0)  # (num_experts,)

            # Router confidence: average probability assigned to each expert
            router_confidence = router_probs.mean(dim=0)  # (num_experts,)

            # Auxiliary loss: minimize correlation between frequency and confidence
            layer_aux_loss = torch.sum(expert_freq * router_confidence) * layer.mlp.num_experts
            total_aux_loss = total_aux_loss + layer_aux_loss

    return total_aux_loss

def compute_moe_loss(model, outputs, router_aux_loss_coef=0.001):
    """
    Compute total loss for MoE model: L_total = L_NTP + λ * L_aux

    Args:
        model: MoE model
        outputs: Model outputs with loss
        router_aux_loss_coef: Coefficient for auxiliary loss (λ)

    Returns:
        total_loss: Combined loss
        ntp_loss: Next token prediction loss
        aux_loss: Auxiliary load balancing loss
    """
    # Get the standard next token prediction loss
    ntp_loss = outputs.loss if hasattr(outputs, 'loss') else None

    # Compute auxiliary loss from all MoE layers
    aux_loss = compute_moe_auxiliary_loss(model, router_aux_loss_coef)

    # Total loss
    if ntp_loss is not None:
        total_loss = ntp_loss + router_aux_loss_coef * aux_loss
    else:
        total_loss = router_aux_loss_coef * aux_loss

    return total_loss, ntp_loss, aux_loss

print("Auxiliary loss computation function defined")


print(" FIXED replace_ffn_with_moe function defined")
print("  - No expert noise - all experts start identical")
print("  - Proper weight extraction and copying")
print("  - Router initialized for top-2 routing")

Auxiliary loss computation function defined
 FIXED replace_ffn_with_moe function defined
  - No expert noise - all experts start identical
  - Proper weight extraction and copying
  - Router initialized for top-2 routing


In [19]:
NUM_EXPERTS = 8  # Full 8 experts like Mixtral
NUM_EXPERTS_PER_TOK = 2  # Top-2 routing like Mixtral
ROUTER_JITTER_NOISE = 0.0
ROUTER_AUX_LOSS_COEF = 0.001  # FIXED: Reduced from 0.01 to allow expert 0 preference initially
USE_HALF_WIDTH = False  # FIXED: Use full intermediate_size (quantization makes this safe)


print("MoE Configuration ")

print(f"  Number of experts: {NUM_EXPERTS}")
print(f"  Experts per token: {NUM_EXPERTS_PER_TOK}")
print(f"  Half-width experts: {USE_HALF_WIDTH} (intermediate_size // 2)")
print(f"  Router jitter noise: {ROUTER_JITTER_NOISE}")
print(f"  Router aux loss coefficient: {ROUTER_AUX_LOSS_COEF}")

# Estimate memory with quantization
intermediate = 14336 // 2 if USE_HALF_WIDTH else 14336  # Full width
expert_params = NUM_EXPERTS * 32 * 3 * 4096 * intermediate  # experts × layers × projections × dims
expert_memory_bf16 = expert_params * 2 / 1e9  # bf16 = 2 bytes
expert_memory_8bit = expert_params * 1 / 1e9  # 8-bit = 1 byte
expert_memory_4bit = expert_params * 0.5 / 1e9  # 4-bit = 0.5 bytes
print(f"\nMemory Estimate:")
print(f"  Intermediate size: {intermediate} {'(half-width)' if USE_HALF_WIDTH else '(full)'}")
print(f"  Expert params: {expert_params/1e9:.2f}B")
print(f"  Expert memory (bf16): ~{expert_memory_bf16:.1f} GB  ← Without quantization (too large!)")
print(f"  Expert memory (8-bit): ~{expert_memory_8bit:.1f} GB  ← With 8-bit quantization")
print(f"  Expert memory (4-bit): ~{expert_memory_4bit:.1f} GB  ← With 4-bit quantization")

# ============================================================================
    #MEMORY CLEANUP BEFORE MOE CREATION
# ============================================================================
import gc
import torch

print("MEMORY CLEANUP BEFORE MOE CREATION")


# Check initial memory
gpu_initial = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory before cleanup: {gpu_initial:.2f} GB")

# Delete base_model if it exists and is not the teacher
if 'base_model' in globals() and 'teacher_model' in globals():
    if base_model is not teacher_model:
        print("  Deleting base_model (keeping teacher_model for KD)")
        del base_model
    else:
        print("  base_model IS teacher_model, keeping it")
elif 'base_model' in globals():
    print("  Deleting base_model")
    del base_model

# Delete any other large model variables
vars_to_delete = ['model']
deleted_count = 0
for var_name in list(globals().keys()):
    if var_name in vars_to_delete and var_name in globals():
        obj = globals().get(var_name)
        if obj is not None and hasattr(obj, 'parameters'):
            try:
                next(obj.parameters())
                print(f"  Deleting: {var_name}")
                del globals()[var_name]
                deleted_count += 1
            except:
                pass

# Aggressive cleanup
for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()
torch.cuda.synchronize()

# Try to trigger malloc_trim
try:
    import ctypes
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except:
    pass

gpu_after_cleanup = torch.cuda.memory_allocated() / 1e9
print(f"\nGPU memory after cleanup: {gpu_after_cleanup:.2f} GB")
print(f"Freed: {gpu_initial - gpu_after_cleanup:.2f} GB")

# CREATE 8-BIT QUANTIZATION CONFIG FOR EXPERTS

from transformers import BitsAndBytesConfig

# FIXED: Enable 8-bit quantization for experts
moe_expert_bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_quant_type="nf4",
)


# LOAD FRESH BASE MODEL FOR MOE CONVERSION


from transformers import AutoModelForCausalLM

base_bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,  
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
    bnb_8bit_quant_type="nf4",
)

print("Loading fresh Mistral-7B model...")
moe_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=base_bnb_config,  # Base model 8-bit
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    )

gpu_with_base = torch.cuda.memory_allocated() / 1e9
print(f" Fresh model loaded")
print(f"  GPU memory: {gpu_with_base:.2f} GB")


print(f"Creating MoE model...")
print(f"  - {NUM_EXPERTS} experts, top-{NUM_EXPERTS_PER_TOK} routing")
print(f"  - Expert precision: 8-bit quantized")
print(f"  - Intermediate size: {intermediate}")
print(f"  - CPU offload: Disabled (breaks gradients)")

moe_model = replace_ffn_with_moe(
    model=moe_model,
    num_experts=NUM_EXPERTS,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK,
    router_jitter_noise=ROUTER_JITTER_NOISE,
    router_aux_loss_coef=ROUTER_AUX_LOSS_COEF,
    bnb_config=moe_expert_bnb_config,  # 8-bit quantized experts
    ram_threshold=80.0,                
    use_disk_offload=True,
    half_width=USE_HALF_WIDTH,
    enable_cpu_offload=False,           # Disabled (breaks training gradients)
)

# Final cleanup
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

gpu_final = torch.cuda.memory_allocated() / 1e9
gpu_reserved = torch.cuda.memory_reserved() / 1e9
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9


print(" MOE MODEL CREATED SUCCESSFULLY")

print(f"  GPU Memory:")
print(f"    Allocated: {gpu_final:.2f} GB")
print(f"    Reserved:  {gpu_reserved:.2f} GB")
print(f"    Total:     {gpu_total:.2f} GB")
print(f"    Usage:     {100*gpu_final/gpu_total:.1f}%")


print(f"    - Experts: {NUM_EXPERTS} ({NUM_EXPERTS_PER_TOK} active per token)")
print(f"    - Expert precision: 8-bit quantized")
print(f"    - Attention layers: 8-bit quantized")
print(f"    - Router: bf16 (trainable)")
print(f"    - Load balancing coefficient: {ROUTER_AUX_LOSS_COEF}")

MoE Configuration 
  Number of experts: 8
  Experts per token: 2
  Half-width experts: False (intermediate_size // 2)
  Router jitter noise: 0.0
  Router aux loss coefficient: 0.001

Memory Estimate:
  Intermediate size: 14336 (full)
  Expert params: 45.10B
  Expert memory (bf16): ~90.2 GB  ← Without quantization (too large!)
  Expert memory (8-bit): ~45.1 GB  ← With 8-bit quantization
  Expert memory (4-bit): ~22.5 GB  ← With 4-bit quantization
MEMORY CLEANUP BEFORE MOE CREATION
GPU memory before cleanup: 7.55 GB
  base_model IS teacher_model, keeping it
  Deleting: model


`torch_dtype` is deprecated! Use `dtype` instead!



GPU memory after cleanup: 7.55 GB
Freed: 0.00 GB
Loading fresh Mistral-7B model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.68s/it]

 Fresh model loaded
  GPU memory: 15.05 GB
Creating MoE model...
  - 8 experts, top-2 routing
  - Expert precision: 8-bit quantized
  - Intermediate size: 14336
  - CPU offload: Disabled (breaks gradients)
Model configuration:
  Hidden size: 4096
  Original intermediate size: 14336
  MoE intermediate size: 14336
  Number of layers: 32
  Experts per layer: 8
  Experts per token: 2

 Using GPU for weight processing
  GPU Memory: 150.11 GB
  Compute dtype: torch.float32

  Processing 32 layers: [0, 1, 2, 3, 4]...

  Layer 1/32 (RAM: 12.3%):
    Extracting weights to CPU... 

Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts (NO NOISE)... Done
    Moving to GPU... Done
  Layer 2/32 (RAM: 12.3%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts (NO NOISE)... Done
    Moving to GPU... Done
  Layer 3/32 (RAM: 12.3%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts (NO NOISE)... Done
    Moving to GPU... Done
  Layer 4/32 (RAM: 12.3%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts (NO NOISE)... Done
    Moving to GPU... Done
    [Layer 4] GPU: 17.52GB alloc / 17.58GB reserved
  Layer 5/32 (RAM: 12.3%):
    Extracting weights to CPU... Done
    Deleting original MLP... Done
    Creating MoE layer... Done
    Copying weights to 8 experts (NO NOISE)...

# Pretraining evaluation for dense MoE

In [20]:
print("MOE BASELINE EVALUATION (not quantized, NO LoRA)")


# 1. MMLU Accuracy
print("\n Evaluating MMLU accuracy...")
moe_results = evaluate_mmlu_comprehensive(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

# 2. FLOPs estimation
print("\n Computing FLOPs...")
moe_flops = compute_model_flops(moe_model, seq_length=MAX_LENGTH)

# 3. Throughput metrics
print(" Measuring throughput...")
moe_throughput = compute_throughput_metrics(
    model=moe_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

# 4. Parameter efficiency
print(" Analyzing parameter efficiency...")
moe_params = compute_parameter_efficiency(
    model=moe_model,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK
)

# 5. Memory metrics
print(" Collecting memory metrics...")
moe_memory = compute_memory_metrics(moe_model)

# Combine all metrics
moe_comprehensive = {
    **moe_results,
    'flops': moe_flops,
    **moe_throughput,
    **moe_params,
    **moe_memory,
}


print("MoE Baseline Evaluation Complete (Pre-LoRA)")
print(f"  Accuracy: {moe_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {moe_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {moe_comprehensive['ece']:.4f}")
print("\n Note: This is the TRUE baseline - no LoRA adapters applied yet")
print("   Next step: Apply QLoRA for training (Cell 15.5.2)")

MOE BASELINE EVALUATION (not quantized, NO LoRA)

 Evaluating MMLU accuracy...


Evaluating: 100%|██████████| 1000/1000 [06:19<00:00,  2.63it/s]



 Computing FLOPs...
 Measuring throughput...
 Analyzing parameter efficiency...
MoE Baseline Evaluation Complete (Pre-LoRA)
  Accuracy: 0.2440
  Top-2 Accuracy: 0.4760
  ECE: 0.2385

 Note: This is the TRUE baseline - no LoRA adapters applied yet
   Next step: Apply QLoRA for training (Cell 15.5.2)


# Training setup for baseline and training 

In [21]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset as TorchDataset
import torch

class MoETrainer(Trainer):
    """
    Custom Trainer for MoE models that computes auxiliary load balancing loss.
    Works with both regular models and PEFT-wrapped models.
    Total loss formula: L_total = L_NTP + λ * L_aux
    """

    def __init__(self, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.router_aux_loss_coef = router_aux_loss_coef

    def _moe_layers(self, model):
        """Get MoE layers from model, handling PEFT wrapper."""
        # Try different model structures (PEFT wrapped, base model, etc.)
        candidates = [
            lambda m: m.model.layers if hasattr(m, 'model') and hasattr(m.model, 'layers') else None,
            lambda m: m.base_model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'layers') else None,
            lambda m: m.base_model.model.model.layers if hasattr(m, 'base_model') and hasattr(m.base_model, 'model') and hasattr(m.base_model.model, 'model') else None,
        ]
        
        for get_layers in candidates:
            try:
                layers = get_layers(model)
                if layers is not None:
                    return layers
            except:
                continue
        return []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Enable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        # Forward pass
        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        
        # Compute auxiliary load balancing loss
        aux_loss = self._compute_moe_auxiliary_loss(model)
        total_loss = ntp_loss + self.router_aux_loss_coef * aux_loss

        # Log losses periodically
        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "train/ntp_loss": ntp_loss.item(),
                "train/aux_loss": aux_loss.item(),
                "train/total_loss": total_loss.item(),
                "train/aux_loss_weight": self.router_aux_loss_coef,
            })

        # Disable router logits collection
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

    def _compute_moe_auxiliary_loss(self, model):
        """Compute auxiliary loss from all MoE layers."""
        device = next(model.parameters()).device
        aux_loss = torch.tensor(0.0, device=device)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()
        return aux_loss

# Training configuration for QLoRA
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": ROUTER_AUX_LOSS_COEF,
    "learning_rate": 2e-4,  # QLoRA standard learning rate
    "batch_size": 4,  # Larger batch with QLoRA
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "output_dir": "./mistral_moe_qlora",
}

print(" MoETrainer class defined (QLoRA compatible)\n")

 MoETrainer class defined (QLoRA compatible)



In [22]:
print("MoETrainer class defined\n")

# ============================================================================
# 15.5.2B - INTEGRATED MOE KD TRAINER
# ============================================================================

import torch.nn.functional as F

class IntegratedMoEKDTrainer(Trainer):
    """
    KD for MoE: L_total = (1-α)*L_NTP + α*L_KD + λ*L_aux + β*L_routingKD
    """
    def __init__(self, teacher_model=None, kd_config=None, router_aux_loss_coef=0.001, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.kd_config = kd_config or {}
        self.router_aux_loss_coef = router_aux_loss_coef
        self.kd_alpha = self.kd_config.get('kd_alpha', 0.5)
        self.temperature = self.kd_config.get('temperature', 4.0)
        self.routing_kd_weight = self.kd_config.get('routing_kd_weight', 0.0)
        self.enable_routing_kd = self.kd_config.get('enable_routing_kd', False)

    def _moe_layers(self, model):
        if hasattr(model, 'model') and hasattr(model.model, 'layers'):
            return model.model.layers
        if hasattr(model, 'base_model') and hasattr(model.base_model.model, 'layers'):
            return model.base_model.model.layers
        return []

    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Compute loss with KD.
        NOTE: Trainer automatically handles device placement - inputs are already on correct device!
        """
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'forward'):
                mlp._collect_router_logits = True

        outputs = model(**inputs)
        ntp_loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        device = ntp_loss.device if hasattr(ntp_loss, 'device') else next(model.parameters()).device

        kd_loss = torch.tensor(0.0, device=device)
        routing_kd_loss = torch.tensor(0.0, device=device)

        if self.teacher_model is not None and self.kd_alpha > 0:
            tdev = next(self.teacher_model.parameters()).device
            tinputs = {k: (v.to(tdev) if hasattr(v, 'to') else v) for k, v in inputs.items()}
            with torch.no_grad():
                t_out = self.teacher_model(**tinputs)

            s_logits = outputs.logits / self.temperature
            t_logits = t_out.logits / self.temperature

            kd_loss = F.kl_div(
                F.log_softmax(s_logits, dim=-1),
                F.softmax(t_logits, dim=-1),
                reduction='batchmean'
            ) * (self.temperature ** 2)

            if self.enable_routing_kd and self.routing_kd_weight > 0:
                ent = torch.tensor(0.0, device=device)
                layers_count = 0
                for layer in self._moe_layers(model):
                    mlp = getattr(layer, 'mlp', None)
                    if mlp is not None and hasattr(mlp, '_last_router_probs') and mlp._last_router_probs is not None:
                        probs = mlp._last_router_probs
                        ent_layer = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean()
                        ent = ent + ent_layer
                        layers_count += 1
                if layers_count > 0:
                    routing_kd_loss = ent / layers_count

        aux_loss = torch.tensor(0.0, device=device)
        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, 'compute_auxiliary_loss'):
                aux_loss = aux_loss + mlp.compute_auxiliary_loss()

        total_loss = ((1 - self.kd_alpha) * ntp_loss
                      + self.kd_alpha * kd_loss
                      + self.router_aux_loss_coef * aux_loss
                      + self.routing_kd_weight * routing_kd_loss)

        for layer in self._moe_layers(model):
            mlp = getattr(layer, 'mlp', None)
            if mlp is not None and hasattr(mlp, '_collect_router_logits'):
                mlp._collect_router_logits = False

        return (total_loss, outputs) if return_outputs else total_loss

print(" IntegratedMoEKDTrainer class defined")

MoETrainer class defined

 IntegratedMoEKDTrainer class defined


In [ ]:
# ============================================================================
# TRAINING CELL - MoE Model Training with Optional Knowledge Distillation
# ============================================================================

import torch
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os

# --------------------------------------------------------------------------
# 1. CONFIGURATION - Choose your training mode
# --------------------------------------------------------------------------

# Training mode selection
USE_KNOWLEDGE_DISTILLATION = False  # Set to True for KD, False for standard training

# Dataset subset configuration
USE_SUBSET = True           # Use only a subset of the data
SUBSET_PERCENTAGE = 0.2     # Use 20% of the dataset

# MoE and training hyperparameters
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": ROUTER_AUX_LOSS_COEF,
    "learning_rate": 2e-4,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "max_steps": 250,  # Train for 250 steps
    "max_length": 512,
}

# LoRA configuration - INCLUDES ROUTERS
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

# Knowledge distillation configuration
KD_CONFIG = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'enable_routing_kd': False,
    'router_aux_loss_coef': TRAINING_CONFIG['router_aux_loss_coef'],
    'name': 'Standard KD',
}

# Output directory
OUTPUT_DIR = "./mistral_moe_qlora_kd" if USE_KNOWLEDGE_DISTILLATION else "./mistral_moe_qlora"


In [ ]:
# --------------------------------------------------------------------------
# 3. SETUP MODELS
# --------------------------------------------------------------------------

student_model = moe_model
student_model.config.use_cache = False

# CRITICAL FIX: Prepare model for k-bit training BEFORE applying LoRA
# This is required when using quantized models with PEFT/LoRA
student_model = prepare_model_for_kbit_training(student_model)
print("✅ Model prepared for k-bit training")

# Enable gradient checkpointing and set to training mode
student_model.gradient_checkpointing_enable()
student_model.train()
print("✅ Model set to training mode with gradient checkpointing")

teacher_model = None
if USE_KNOWLEDGE_DISTILLATION:
    print("\nLoading teacher model (baseline full-width model)...")
    teacher_model = base_model
    teacher_model.eval()
    for param in teacher_model.parameters():
        param.requires_grad = False
    print("Teacher model loaded and frozen.")


In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
# --------------------------------------------------------------------------
# 4. FIX TOKENIZER - Set padding token
# --------------------------------------------------------------------------

print("\nConfiguring tokenizer...")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"Set pad_token to eos_token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
else:
    print(f"Tokenizer already has pad_token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

tokenizer.padding_side = "left"
print(f"Padding side: {tokenizer.padding_side}")

if hasattr(student_model, 'config'):
    student_model.config.pad_token_id = tokenizer.pad_token_id
    print(f"Updated model config pad_token_id: {student_model.config.pad_token_id}")

# --------------------------------------------------------------------------
# 5. CREATE DATASET SUBSET (20%)
# --------------------------------------------------------------------------

if USE_SUBSET:
    print(f"\nCreating {SUBSET_PERCENTAGE*100}% subset of training data...")
    original_size = len(train_dataset_raw)
    subset_size = int(original_size * SUBSET_PERCENTAGE)

    train_dataset_subset = train_dataset_raw.select(range(subset_size))

    print(f"Original dataset size: {original_size:,}")
    print(f"Subset dataset size: {len(train_dataset_subset):,}")

    train_dataset_to_use = train_dataset_subset
else:
    train_dataset_to_use = train_dataset_raw
    print(f"Using full training dataset: {len(train_dataset_to_use):,} samples")

# --------------------------------------------------------------------------
# 6. TOKENIZE DATASET
# --------------------------------------------------------------------------

print("\nTokenizing dataset...")

def tokenize_function(examples):
    """Tokenize for causal language modeling."""
    return tokenizer(
        examples['formatted_prompt'],
        truncation=True,
        max_length=TRAINING_CONFIG['max_length'],
        padding=False,
    )

print("Tokenizing training dataset...")
train_dataset_tokenized = train_dataset_to_use.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset_to_use.column_names,
    desc="Tokenizing training data"
)

print(f"Training dataset tokenized: {len(train_dataset_tokenized):,} samples")

eval_dataset_tokenized = None
if 'eval_dataset' in dir():
    print("\nTokenizing evaluation dataset...")

    if USE_SUBSET:
        eval_subset_size = int(len(eval_dataset) * SUBSET_PERCENTAGE)
        eval_dataset_subset = eval_dataset.select(range(eval_subset_size))
        print(f"Using {len(eval_dataset_subset):,} eval samples")
    else:
        eval_dataset_subset = eval_dataset

    eval_dataset_tokenized = eval_dataset_subset.map(
        tokenize_function,
        batched=True,
        remove_columns=eval_dataset_subset.column_names,
        desc="Tokenizing eval data"
    )
    print(f"Eval dataset tokenized: {len(eval_dataset_tokenized):,} samples")

# --------------------------------------------------------------------------
# 7. SETUP DATA COLLATOR
# --------------------------------------------------------------------------

print("\nSetting up data collator...")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Data collator configured")


# Model addaptation and LoRA implementation


In [ ]:
# --------------------------------------------------------------------------
# 8. BUILD TARGET MODULES LIST FOR LORA (INCLUDING ROUTERS!)
# --------------------------------------------------------------------------

print("\nBuilding target modules list for LoRA...")

base_model_ref = student_model
if hasattr(student_model, 'model'):
    base_model_ref = student_model.model

num_layers = len(base_model_ref.layers)
print(f"Number of layers: {num_layers}")

target_modules = []

# Attention layers
attention_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
target_modules.extend(attention_modules)

# Expert layers - individual paths to each expert
expert_types = ["gate_proj", "up_proj", "down_proj"]
num_experts = 0

for layer_idx in range(num_layers):
    layer = base_model_ref.layers[layer_idx]

    if hasattr(layer.mlp, 'gate_proj') and isinstance(layer.mlp.gate_proj, torch.nn.ModuleList):
        num_experts = len(layer.mlp.gate_proj)

        # Add expert layers
        for expert_type in expert_types:
            for expert_idx in range(num_experts):
                target_modules.append(f"model.layers.{layer_idx}.mlp.{expert_type}.{expert_idx}")

        # CRITICAL: Add router layer with LoRA adapters
        target_modules.append(f"model.layers.{layer_idx}.mlp.router")

print(f"\nLoRA will target:")
print(f"  Attention: {len(attention_modules)} module types x {num_layers} layers")
print(f"  Experts: {len(expert_types)} types x {num_experts} experts x {num_layers} layers")
print(f"  Routers: {num_layers} router layers (via LoRA)")
print(f"  Total LoRA targets: {len(target_modules)}")

# --------------------------------------------------------------------------
# 9. APPLY LORA
# --------------------------------------------------------------------------

print("\nApplying LoRA to model (this will take a moment)...")

lora_config = LoraConfig(
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    target_modules=target_modules,
    lora_dropout=LORA_CONFIG["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

student_model = get_peft_model(student_model, lora_config)
print("✅ LoRA applied successfully")

# --------------------------------------------------------------------------
# 10. VERIFY TRAINABLE PARAMETERS
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINABLE PARAMETERS VERIFICATION")
print("=" * 80)

# Get counts
total_params = sum(p.numel() for p in student_model.parameters())
trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad)

# Detailed breakdown
lora_attention_params = 0
lora_expert_params = 0
lora_router_params = 0
other_trainable = 0

trainable_components = {
    'lora_attention': [],
    'lora_experts': [],
    'lora_routers': [],
    'other': []
}

for name, param in student_model.named_parameters():
    if param.requires_grad:
        if 'lora' in name.lower():
            if any(attn in name for attn in ['q_proj', 'k_proj', 'v_proj', 'o_proj']):
                lora_attention_params += param.numel()
                trainable_components['lora_attention'].append((name, param.numel()))
            elif any(exp in name for exp in ['gate_proj', 'up_proj', 'down_proj']):
                lora_expert_params += param.numel()
                trainable_components['lora_experts'].append((name, param.numel()))
            elif 'router' in name:
                lora_router_params += param.numel()
                trainable_components['lora_routers'].append((name, param.numel()))
            else:
                other_trainable += param.numel()
                trainable_components['other'].append((name, param.numel()))
        else:
            other_trainable += param.numel()
            trainable_components['other'].append((name, param.numel()))

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print(f"\nBreakdown:")
print(f"  LoRA attention adapters: {lora_attention_params:,} ({len(trainable_components['lora_attention'])} adapters)")
print(f"  LoRA expert adapters: {lora_expert_params:,} ({len(trainable_components['lora_experts'])} adapters)")
print(f"  LoRA router adapters: {lora_router_params:,} ({len(trainable_components['lora_routers'])} adapters)")
print(f"  Other: {other_trainable:,}")

# Show sample trainable parameters
print(f"\nSample trainable parameters:")
for component, params_list in trainable_components.items():
    if params_list:
        print(f"\n{component.upper()}:")
        for name, count in params_list[:3]:
            print(f"  {name}: {count:,} params")
        if len(params_list) > 3:
            print(f"  ... and {len(params_list) - 3} more")

# CRITICAL CHECK
if trainable_params == 0:
    raise RuntimeError("FATAL ERROR: No trainable parameters found!")

print("\n" + "=" * 80)
print("WHAT GETS TRAINED:")
print("=" * 80)
print("✅ Attention layers (q/k/v/o_proj): LoRA adapters")
print("✅ Expert FFNs (gate/up/down_proj): LoRA adapters on all 8 experts")
print("✅ Routers: LoRA adapters (low-rank updates to routing)")
print("❌ Base model weights: Frozen (8-bit quantized)")
print("\nThis is full QLoRA training - experts, attention, AND routing!")
print("=" * 80)


In [46]:
# ============================================================================
# TRAINING CELL - MoE Model Training with Optional Knowledge Distillation
# ============================================================================

import torch
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import os

# --------------------------------------------------------------------------
# 1. CONFIGURATION - Choose your training mode
# --------------------------------------------------------------------------

# Training mode selection
USE_KNOWLEDGE_DISTILLATION = False  # Set to True for KD, False for standard training

# Dataset subset configuration
USE_SUBSET = True           # Use only a subset of the data
SUBSET_PERCENTAGE = 0.2     # Use 20% of the dataset

# MoE and training hyperparameters
TRAINING_CONFIG = {
    "num_experts": NUM_EXPERTS,
    "num_experts_per_tok": NUM_EXPERTS_PER_TOK,
    "router_jitter_noise": ROUTER_JITTER_NOISE,
    "router_aux_loss_coef": ROUTER_AUX_LOSS_COEF,
    "learning_rate": 2e-4,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "num_train_epochs": 1,
    "logging_steps": 25,
    "eval_steps": 100,
    "save_steps": 100,
    "max_steps": 250,  # Train for 250 steps
    "max_length": 512,
}

# LoRA configuration - INCLUDES ROUTERS
LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

# Knowledge distillation configuration
KD_CONFIG = {
    'kd_alpha': 0.5,
    'temperature': 4.0,
    'routing_kd_weight': 0.0,
    'enable_routing_kd': False,
    'router_aux_loss_coef': TRAINING_CONFIG['router_aux_loss_coef'],
    'name': 'Standard KD',
}

# Output directory
OUTPUT_DIR = "./mistral_moe_qlora_kd" if USE_KNOWLEDGE_DISTILLATION else "./mistral_moe_qlora"

# --------------------------------------------------------------------------
# 2. VERIFY REQUIRED VARIABLES
# --------------------------------------------------------------------------

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

required_vars = {
    'moe_model': 'MoE student model',
    'tokenizer': 'Tokenizer',
    'train_dataset_raw': 'Training dataset',
}

missing_vars = []
for var_name, description in required_vars.items():
    if var_name not in dir():
        missing_vars.append(f"  - {var_name} ({description})")

if missing_vars:
    print("ERROR: The following required variables are not defined:")
    print("\n".join(missing_vars))
    print("\nPlease run the appropriate cells to define these variables before training.")
    raise NameError(f"Missing required variables: {', '.join([v.split()[1] for v in missing_vars])}")

if USE_KNOWLEDGE_DISTILLATION and 'base_model' not in dir():
    print("ERROR: Knowledge Distillation is enabled but 'base_model' is not defined.")
    print("Please load the baseline full-width model first, or set USE_KNOWLEDGE_DISTILLATION=False")
    raise NameError("base_model is required for knowledge distillation")

print(f"Training Mode: {'Knowledge Distillation' if USE_KNOWLEDGE_DISTILLATION else 'Standard Training'}")
print(f"Model: {model_id}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Number of Experts: {TRAINING_CONFIG['num_experts']}")
print(f"Experts per Token: {TRAINING_CONFIG['num_experts_per_tok']}")
print(f"Learning Rate: {TRAINING_CONFIG['learning_rate']}")
print(f"Batch Size: {TRAINING_CONFIG['batch_size']} (effective: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']})")
print(f"Max Steps: {TRAINING_CONFIG['max_steps']}")
print("=" * 80)

# --------------------------------------------------------------------------
# 3. SETUP MODELS
# --------------------------------------------------------------------------

student_model = moe_model
student_model.config.use_cache = False

# CRITICAL FIX: Prepare model for k-bit training BEFORE applying LoRA
# This is required when using quantized models with PEFT/LoRA
student_model = prepare_model_for_kbit_training(student_model)
print("✅ Model prepared for k-bit training")

# Enable gradient checkpointing and set to training mode
student_model.gradient_checkpointing_enable()
student_model.train()
print("✅ Model set to training mode with gradient checkpointing")

teacher_model = None
if USE_KNOWLEDGE_DISTILLATION:
    print("\nLoading teacher model (baseline full-width model)...")
    teacher_model = base_model
    teacher_model.eval()
    for param in teacher_model.parameters():
        param.requires_grad = False
    print("Teacher model loaded and frozen.")

# --------------------------------------------------------------------------
# 4. FIX TOKENIZER - Set padding token
# --------------------------------------------------------------------------

print("\nConfiguring tokenizer...")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"Set pad_token to eos_token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
else:
    print(f"Tokenizer already has pad_token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

tokenizer.padding_side = "left"
print(f"Padding side: {tokenizer.padding_side}")

if hasattr(student_model, 'config'):
    student_model.config.pad_token_id = tokenizer.pad_token_id
    print(f"Updated model config pad_token_id: {student_model.config.pad_token_id}")

# --------------------------------------------------------------------------
# 5. CREATE DATASET SUBSET (20%)
# --------------------------------------------------------------------------

if USE_SUBSET:
    print(f"\nCreating {SUBSET_PERCENTAGE*100}% subset of training data...")
    original_size = len(train_dataset_raw)
    subset_size = int(original_size * SUBSET_PERCENTAGE)

    train_dataset_subset = train_dataset_raw.select(range(subset_size))

    print(f"Original dataset size: {original_size:,}")
    print(f"Subset dataset size: {len(train_dataset_subset):,}")

    train_dataset_to_use = train_dataset_subset
else:
    train_dataset_to_use = train_dataset_raw
    print(f"Using full training dataset: {len(train_dataset_to_use):,} samples")

# --------------------------------------------------------------------------
# 6. TOKENIZE DATASET
# --------------------------------------------------------------------------

print("\nTokenizing dataset...")

def tokenize_function(examples):
    """Tokenize for causal language modeling."""
    return tokenizer(
        examples['formatted_prompt'],
        truncation=True,
        max_length=TRAINING_CONFIG['max_length'],
        padding=False,
    )

print("Tokenizing training dataset...")
train_dataset_tokenized = train_dataset_to_use.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset_to_use.column_names,
    desc="Tokenizing training data"
)

print(f"Training dataset tokenized: {len(train_dataset_tokenized):,} samples")

eval_dataset_tokenized = None
if 'eval_dataset' in dir():
    print("\nTokenizing evaluation dataset...")

    if USE_SUBSET:
        eval_subset_size = int(len(eval_dataset) * SUBSET_PERCENTAGE)
        eval_dataset_subset = eval_dataset.select(range(eval_subset_size))
        print(f"Using {len(eval_dataset_subset):,} eval samples")
    else:
        eval_dataset_subset = eval_dataset

    eval_dataset_tokenized = eval_dataset_subset.map(
        tokenize_function,
        batched=True,
        remove_columns=eval_dataset_subset.column_names,
        desc="Tokenizing eval data"
    )
    print(f"Eval dataset tokenized: {len(eval_dataset_tokenized):,} samples")

# --------------------------------------------------------------------------
# 7. SETUP DATA COLLATOR
# --------------------------------------------------------------------------

print("\nSetting up data collator...")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Data collator configured")

# --------------------------------------------------------------------------
# 8. BUILD TARGET MODULES LIST FOR LORA (INCLUDING ROUTERS!)
# --------------------------------------------------------------------------

print("\nBuilding target modules list for LoRA...")

base_model_ref = student_model
if hasattr(student_model, 'model'):
    base_model_ref = student_model.model

num_layers = len(base_model_ref.layers)
print(f"Number of layers: {num_layers}")

target_modules = []

# Attention layers
attention_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
target_modules.extend(attention_modules)

# Expert layers - individual paths to each expert
expert_types = ["gate_proj", "up_proj", "down_proj"]
num_experts = 0

for layer_idx in range(num_layers):
    layer = base_model_ref.layers[layer_idx]

    if hasattr(layer.mlp, 'gate_proj') and isinstance(layer.mlp.gate_proj, torch.nn.ModuleList):
        num_experts = len(layer.mlp.gate_proj)

        # Add expert layers
        for expert_type in expert_types:
            for expert_idx in range(num_experts):
                target_modules.append(f"model.layers.{layer_idx}.mlp.{expert_type}.{expert_idx}")

        # CRITICAL: Add router layer with LoRA adapters
        target_modules.append(f"model.layers.{layer_idx}.mlp.router")

print(f"\nLoRA will target:")
print(f"  Attention: {len(attention_modules)} module types x {num_layers} layers")
print(f"  Experts: {len(expert_types)} types x {num_experts} experts x {num_layers} layers")
print(f"  Routers: {num_layers} router layers (via LoRA)")
print(f"  Total LoRA targets: {len(target_modules)}")

# --------------------------------------------------------------------------
# 9. APPLY LORA
# --------------------------------------------------------------------------

print("\nApplying LoRA to model (this will take a moment)...")

lora_config = LoraConfig(
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    target_modules=target_modules,
    lora_dropout=LORA_CONFIG["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

student_model = get_peft_model(student_model, lora_config)
print("✅ LoRA applied successfully")

# --------------------------------------------------------------------------
# 10. VERIFY TRAINABLE PARAMETERS
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINABLE PARAMETERS VERIFICATION")
print("=" * 80)

# Get counts
total_params = sum(p.numel() for p in student_model.parameters())
trainable_params = sum(p.numel() for p in student_model.parameters() if p.requires_grad)

# Detailed breakdown
lora_attention_params = 0
lora_expert_params = 0
lora_router_params = 0
other_trainable = 0

trainable_components = {
    'lora_attention': [],
    'lora_experts': [],
    'lora_routers': [],
    'other': []
}

for name, param in student_model.named_parameters():
    if param.requires_grad:
        if 'lora' in name.lower():
            if any(attn in name for attn in ['q_proj', 'k_proj', 'v_proj', 'o_proj']):
                lora_attention_params += param.numel()
                trainable_components['lora_attention'].append((name, param.numel()))
            elif any(exp in name for exp in ['gate_proj', 'up_proj', 'down_proj']):
                lora_expert_params += param.numel()
                trainable_components['lora_experts'].append((name, param.numel()))
            elif 'router' in name:
                lora_router_params += param.numel()
                trainable_components['lora_routers'].append((name, param.numel()))
            else:
                other_trainable += param.numel()
                trainable_components['other'].append((name, param.numel()))
        else:
            other_trainable += param.numel()
            trainable_components['other'].append((name, param.numel()))

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print(f"\nBreakdown:")
print(f"  LoRA attention adapters: {lora_attention_params:,} ({len(trainable_components['lora_attention'])} adapters)")
print(f"  LoRA expert adapters: {lora_expert_params:,} ({len(trainable_components['lora_experts'])} adapters)")
print(f"  LoRA router adapters: {lora_router_params:,} ({len(trainable_components['lora_routers'])} adapters)")
print(f"  Other: {other_trainable:,}")

# Show sample trainable parameters
print(f"\nSample trainable parameters:")
for component, params_list in trainable_components.items():
    if params_list:
        print(f"\n{component.upper()}:")
        for name, count in params_list[:3]:
            print(f"  {name}: {count:,} params")
        if len(params_list) > 3:
            print(f"  ... and {len(params_list) - 3} more")

# CRITICAL CHECK
if trainable_params == 0:
    raise RuntimeError("FATAL ERROR: No trainable parameters found!")

print("\n" + "=" * 80)
print("WHAT GETS TRAINED:")
print("=" * 80)
print("✅ Attention layers (q/k/v/o_proj): LoRA adapters")
print("✅ Expert FFNs (gate/up/down_proj): LoRA adapters on all 8 experts")
print("✅ Routers: LoRA adapters (low-rank updates to routing)")
print("❌ Base model weights: Frozen (8-bit quantized)")
print("\nThis is full QLoRA training - experts, attention, AND routing!")
print("=" * 80)

# --------------------------------------------------------------------------
# 11. SETUP TRAINING ARGUMENTS
# --------------------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=TRAINING_CONFIG['num_train_epochs'],
    max_steps=TRAINING_CONFIG['max_steps'],
    per_device_train_batch_size=TRAINING_CONFIG['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
    warmup_ratio=TRAINING_CONFIG['warmup_ratio'],
    learning_rate=TRAINING_CONFIG['learning_rate'],

    optim="adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,

    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=TRAINING_CONFIG['logging_steps'],
    eval_strategy="steps" if eval_dataset_tokenized is not None else "no",
    eval_steps=TRAINING_CONFIG['eval_steps'] if eval_dataset_tokenized is not None else None,
    save_strategy="steps",
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=3,

    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,

    load_best_model_at_end=False,
    report_to="tensorboard",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
)

# --------------------------------------------------------------------------
# 12. INITIALIZE TRAINER
# --------------------------------------------------------------------------

print("\nInitializing trainer...")

if USE_KNOWLEDGE_DISTILLATION:
    trainer = IntegratedMoEKDTrainer(
        model=student_model,
        teacher_model=teacher_model,
        kd_config=KD_CONFIG,
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        args=training_args,
        train_dataset=train_dataset_tokenized,
        eval_dataset=eval_dataset_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    print(f"✅ KD Trainer initialized: {KD_CONFIG['name']}")
else:
    trainer = MoETrainer(
        router_aux_loss_coef=TRAINING_CONFIG['router_aux_loss_coef'],
        model=student_model,
        args=training_args,
        train_dataset=train_dataset_tokenized,
        eval_dataset=eval_dataset_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    print("✅ Standard MoE Trainer initialized")

# --------------------------------------------------------------------------
# 13. START TRAINING
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)
print(f"Max steps: {TRAINING_CONFIG['max_steps']}")
print(f"Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"Training samples: {len(train_dataset_tokenized):,}")
print("=" * 80)

torch.cuda.empty_cache()

trainer.train()

print("\n" + "=" * 80)
print("TRAINING COMPLETED")
print("=" * 80)

# --------------------------------------------------------------------------
# 14. SAVE FINAL MODEL
# --------------------------------------------------------------------------

print("\nSaving final model...")
trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
print(f"✅ Model saved to: {OUTPUT_DIR}/final_model")

# --------------------------------------------------------------------------
# 15. TRAINING SUMMARY
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING SUMMARY")
print("=" * 80)
print(f"Mode: {'Knowledge Distillation' if USE_KNOWLEDGE_DISTILLATION else 'Standard'}")
print(f"Dataset: {len(train_dataset_tokenized):,} samples (20% subset)")
print(f"Steps completed: {trainer.state.global_step}")
print(f"Final loss: {trainer.state.log_history[-1].get('loss', 'N/A') if trainer.state.log_history else 'N/A'}")
print(f"Model location: {OUTPUT_DIR}/final_model")
print("=" * 80)

torch.cuda.empty_cache()


TRAINING CONFIGURATION
Training Mode: Standard Training
Model: mistralai/Mistral-7B-v0.1
Output Directory: ./mistral_moe_qlora
Number of Experts: 8
Experts per Token: 2
Learning Rate: 0.0002
Batch Size: 4 (effective: 16)
Max Steps: 250
✅ Model prepared for k-bit training
✅ Model set to training mode with gradient checkpointing

Configuring tokenizer...
Tokenizer already has pad_token: '</s>' (ID: 2)
Padding side: left
Updated model config pad_token_id: 2

Creating 20.0% subset of training data...
Original dataset size: 68,940
Subset dataset size: 13,788

Tokenizing dataset...
Tokenizing training dataset...


Tokenizing training data: 100%|██████████| 13788/13788 [00:05<00:00, 2541.60 examples/s]


Training dataset tokenized: 13,788 samples

Tokenizing evaluation dataset...
Using 5,909 eval samples


Tokenizing eval data: 100%|██████████| 5909/5909 [00:02<00:00, 2522.81 examples/s]


Eval dataset tokenized: 5,909 samples

Setting up data collator...
Data collator configured

Building target modules list for LoRA...
Number of layers: 32

LoRA will target:
  Attention: 4 module types x 32 layers
  Experts: 3 types x 8 experts x 32 layers
  Routers: 32 router layers (via LoRA)
  Total LoRA targets: 804

Applying LoRA to model (this will take a moment)...
✅ LoRA applied successfully

TRAINABLE PARAMETERS VERIFICATION
Total parameters: 24,396,439,552
Trainable parameters: 242,225,152
Trainable percentage: 0.9929%

Breakdown:
  LoRA attention adapters: 13,631,488 (256 adapters)
  LoRA expert adapters: 226,492,416 (1536 adapters)
  LoRA router adapters: 2,101,248 (64 adapters)
  Other: 0

Sample trainable parameters:

LORA_ATTENTION:
  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: 65,536 params
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: 65,536 params
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.

Step,Training Loss,Validation Loss
100,28.672700,7.007941
200,26.643800,6.523429



TRAINING COMPLETED

Saving final model...
✅ Model saved to: ./mistral_moe_qlora/final_model

TRAINING SUMMARY
Mode: Standard
Dataset: 13,788 samples (20% subset)
Steps completed: 250
Final loss: N/A
Model location: ./mistral_moe_qlora/final_model


In [49]:
# ============================================================================
# TRAINED MoE MODEL COMPREHENSIVE EVALUATION
# ============================================================================

import json
import os
import numpy as np

print("=" * 80)
print("TRAINED MoE MODEL COMPREHENSIVE EVALUATION")
print("=" * 80)

# --------------------------------------------------------------------------
# 1. SETUP - Load trained model
# --------------------------------------------------------------------------

print("\n📦 Loading trained model...")

# Determine which model to evaluate
if 'student_model' in dir() and student_model is not None:
    trained_model = student_model
    model_name = "Trained MoE Model (in-memory)"
    print("✅ Using student_model from training session")
else:
    # Load from checkpoint if not in memory
    from peft import PeftModel
    
    # Try to load from the most recent checkpoint
    checkpoint_dir = OUTPUT_DIR if 'OUTPUT_DIR' in dir() else "./mistral_moe_qlora"
    
    print(f"Loading from checkpoint: {checkpoint_dir}/final_model")
    
    # Load base MoE model first
    trained_model = moe_model  # Assumes moe_model is still in memory
    
    # Load LoRA weights
    trained_model = PeftModel.from_pretrained(trained_model, f"{checkpoint_dir}/final_model")
    model_name = f"Trained Model from {checkpoint_dir}"
    print(f"✅ Loaded checkpoint: {checkpoint_dir}/final_model")

# Set to eval mode
trained_model.eval()
print(f"✅ Model set to evaluation mode")

# --------------------------------------------------------------------------
# 2. MMLU Accuracy
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. MMLU ACCURACY EVALUATION")
print("=" * 80)

trained_results = evaluate_mmlu_comprehensive(
    model=trained_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    answer_tokens=ANSWER_TOKENS,
    device="cuda",
    max_samples=1000,
    show_progress=True
)

print(f"✅ MMLU Accuracy: {trained_results['accuracy']:.4f}")
print(f"✅ Top-2 Accuracy: {trained_results['top2_accuracy']:.4f}")
print(f"✅ ECE: {trained_results['ece']:.4f}")

# --------------------------------------------------------------------------
# 3. FLOPs Estimation
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. COMPUTATIONAL EFFICIENCY (FLOPs)")
print("=" * 80)

print("Computing FLOPs...")
trained_flops = compute_model_flops(trained_model, seq_length=MAX_LENGTH)
print(f"✅ FLOPs per forward pass: {trained_flops/1e9:.2f}G")

# --------------------------------------------------------------------------
# 4. Throughput Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. THROUGHPUT METRICS")
print("=" * 80)

print("Measuring throughput...")
trained_throughput = compute_throughput_metrics(
    model=trained_model,
    tokenizer=tokenizer,
    eval_dataset=eval_dataset,
    max_samples=100
)

print(f"✅ Tokens/second: {trained_throughput['tokens_per_second']:.2f}")
print(f"✅ ms/token: {trained_throughput['ms_per_token']:.2f}")
print(f"✅ Samples/second: {trained_throughput['samples_per_second']:.2f}")

# --------------------------------------------------------------------------
# 5. Parameter Efficiency
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. PARAMETER EFFICIENCY")
print("=" * 80)

print("Analyzing parameter efficiency...")

# Determine num_experts_per_tok from model
base_model_ref = trained_model
if hasattr(trained_model, 'base_model'):
    base_model_ref = trained_model.base_model
if hasattr(base_model_ref, 'model'):
    base_model_ref = base_model_ref.model

# Get num_experts_per_tok from first MoE layer
num_experts_per_tok_value = 2  # Default
if hasattr(base_model_ref, 'layers') and len(base_model_ref.layers) > 0:
    first_layer = base_model_ref.layers[0]
    if hasattr(first_layer, 'mlp') and hasattr(first_layer.mlp, 'num_experts_per_tok'):
        num_experts_per_tok_value = first_layer.mlp.num_experts_per_tok

trained_params = compute_parameter_efficiency(
    model=trained_model,
    num_experts_per_tok=num_experts_per_tok_value
)

print(f"✅ Total parameters: {trained_params['total_params']/1e9:.2f}B")
print(f"✅ Active parameters: {trained_params['active_params']/1e9:.2f}B")
print(f"✅ Trainable parameters: {trained_params['trainable_params']/1e6:.2f}M")
print(f"✅ Sparsity ratio: {trained_params['sparsity_ratio']:.2%}")

# --------------------------------------------------------------------------
# 6. Memory Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. MEMORY USAGE")
print("=" * 80)

print("Collecting memory metrics...")
trained_memory = compute_memory_metrics(trained_model)

print(f"✅ Model size: {trained_memory['model_size_mb']:.2f} MB")
print(f"✅ GPU allocated: {trained_memory['gpu_memory_allocated_gb']:.2f} GB")
print(f"✅ GPU reserved: {trained_memory['gpu_memory_reserved_gb']:.2f} GB")

# --------------------------------------------------------------------------
# 7. Combine All Metrics
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMBINING ALL METRICS")
print("=" * 80)

trained_comprehensive = {
    **trained_results,
    'flops': trained_flops,
    **trained_throughput,
    **trained_params,
    **trained_memory,
}

# --------------------------------------------------------------------------
# 8. Display Comprehensive Results
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPREHENSIVE TRAINED MoE METRICS")
print("=" * 80)

print("\n📊 Accuracy Metrics:")
print(f"  MMLU Accuracy: {trained_comprehensive['accuracy']:.4f}")
print(f"  Top-2 Accuracy: {trained_comprehensive['top2_accuracy']:.4f}")
print(f"  ECE: {trained_comprehensive['ece']:.4f}")

print("\n⚡ Computational Efficiency:")
print(f"  FLOPs per forward pass: {trained_comprehensive['flops']/1e9:.2f}G")
print(f"  Tokens/second: {trained_comprehensive['tokens_per_second']:.2f}")
print(f"  ms/token: {trained_comprehensive['ms_per_token']:.2f}")
print(f"  Samples/second: {trained_comprehensive['samples_per_second']:.2f}")

print("\n🔧 Parameter Efficiency:")
print(f"  Total parameters: {trained_comprehensive['total_params']/1e9:.2f}B")
print(f"  Active parameters: {trained_comprehensive['active_params']/1e9:.2f}B")
print(f"  Trainable parameters: {trained_comprehensive['trainable_params']/1e6:.2f}M")
print(f"  Sparsity ratio: {trained_comprehensive['sparsity_ratio']:.2%}")

print("\n💾 Memory Usage:")
print(f"  Model size: {trained_comprehensive['model_size_mb']:.2f} MB")
print(f"  GPU allocated: {trained_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
print(f"  GPU reserved: {trained_comprehensive['gpu_memory_reserved_gb']:.2f} GB")

# --------------------------------------------------------------------------
# 9. Compare with Baseline (if available)
# --------------------------------------------------------------------------

if 'baseline_comprehensive' in dir():
    print("\n" + "=" * 80)
    print("COMPARISON WITH BASELINE")
    print("=" * 80)
    
    print("\n📈 Accuracy Comparison:")
    acc_diff = trained_comprehensive['accuracy'] - baseline_comprehensive['accuracy']
    acc_change = (acc_diff / baseline_comprehensive['accuracy']) * 100
    print(f"  Baseline Accuracy:     {baseline_comprehensive['accuracy']:.4f}")
    print(f"  Trained Accuracy:      {trained_comprehensive['accuracy']:.4f}")
    print(f"  Difference:            {acc_diff:+.4f} ({acc_change:+.2f}%)")
    
    print("\n⚡ Efficiency Comparison:")
    throughput_diff = trained_comprehensive['tokens_per_second'] - baseline_comprehensive['tokens_per_second']
    throughput_change = (throughput_diff / baseline_comprehensive['tokens_per_second']) * 100
    print(f"  Baseline Tokens/sec:   {baseline_comprehensive['tokens_per_second']:.2f}")
    print(f"  Trained Tokens/sec:    {trained_comprehensive['tokens_per_second']:.2f}")
    print(f"  Difference:            {throughput_diff:+.2f} ({throughput_change:+.2f}%)")
    
    print("\n💾 Memory Comparison:")
    mem_diff = trained_comprehensive['gpu_memory_allocated_gb'] - baseline_comprehensive['gpu_memory_allocated_gb']
    print(f"  Baseline GPU Memory:   {baseline_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  Trained GPU Memory:    {trained_comprehensive['gpu_memory_allocated_gb']:.2f} GB")
    print(f"  Difference:            {mem_diff:+.2f} GB")
    
    print("\n🎯 Parameter Efficiency:")
    active_ratio_baseline = baseline_comprehensive['active_params'] / baseline_comprehensive['total_params']
    active_ratio_trained = trained_comprehensive['active_params'] / trained_comprehensive['total_params']
    print(f"  Baseline Active/Total: {active_ratio_baseline:.2%}")
    print(f"  Trained Active/Total:  {active_ratio_trained:.2%}")

# --------------------------------------------------------------------------
# 10. Save Results
# --------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

os.makedirs("results", exist_ok=True)

# Save trained model results
results_file = "results/trained_moe_comprehensive.json"
with open(results_file, 'w') as f:
    json.dump({k: v for k, v in trained_comprehensive.items()
               if not isinstance(v, np.ndarray)}, f, indent=2, default=str)
print(f"✅ Saved trained model results to: {results_file}")

# Save comparison if baseline exists
if 'baseline_comprehensive' in dir():
    comparison_results = {
        'baseline': {k: v for k, v in baseline_comprehensive.items() if not isinstance(v, np.ndarray)},
        'trained': {k: v for k, v in trained_comprehensive.items() if not isinstance(v, np.ndarray)},
        'improvements': {
            'accuracy_diff': acc_diff,
            'accuracy_change_pct': acc_change,
            'throughput_diff': throughput_diff,
            'throughput_change_pct': throughput_change,
            'memory_diff_gb': mem_diff,
        }
    }
    
    comparison_file = "results/baseline_vs_trained_comparison.json"
    with open(comparison_file, 'w') as f:
        json.dump(comparison_results, f, indent=2, default=str)
    print(f"✅ Saved comparison results to: {comparison_file}")

# --------------------------------------------------------------------------
# 11. Log to WandB (if available)
# --------------------------------------------------------------------------

try:
    if 'wandb' in dir() and wandb.run is not None:
        print("\n📊 Logging to WandB...")
        wandb.log({
            'trained/accuracy': trained_comprehensive['accuracy'],
            'trained/top2_accuracy': trained_comprehensive['top2_accuracy'],
            'trained/ece': trained_comprehensive['ece'],
            'trained/flops_billions': trained_comprehensive['flops']/1e9,
            'trained/tokens_per_second': trained_comprehensive['tokens_per_second'],
            'trained/ms_per_token': trained_comprehensive['ms_per_token'],
            'trained/active_params_billions': trained_comprehensive['active_params']/1e9,
            'trained/trainable_params_millions': trained_comprehensive['trainable_params']/1e6,
            'trained/gpu_memory_gb': trained_comprehensive['gpu_memory_allocated_gb'],
            'trained/sparsity_ratio': trained_comprehensive['sparsity_ratio'],
        })
        
        # Log comparison if baseline exists
        if 'baseline_comprehensive' in dir():
            wandb.log({
                'comparison/accuracy_improvement': acc_diff,
                'comparison/accuracy_improvement_pct': acc_change,
                'comparison/throughput_improvement': throughput_diff,
                'comparison/throughput_improvement_pct': throughput_change,
            })
        
        print("✅ Logged to WandB")
except Exception as e:
    print(f"⚠️  Could not log to WandB: {e}")

# --------------------------------------------------------------------------
# 12. Store for later use
# --------------------------------------------------------------------------

trained_accuracy = trained_comprehensive['accuracy']

print("\n" + "=" * 80)
print("✅ COMPREHENSIVE TRAINED MoE EVALUATION COMPLETE!")
print("=" * 80)

# Clean up
torch.cuda.empty_cache()

TRAINED MoE MODEL COMPREHENSIVE EVALUATION

📦 Loading trained model...
✅ Using student_model from training session
✅ Model set to evaluation mode

1. MMLU ACCURACY EVALUATION


Evaluating: 100%|██████████| 1000/1000 [04:12<00:00,  3.97it/s]


✅ MMLU Accuracy: 0.2330
✅ Top-2 Accuracy: 0.4850
✅ ECE: 0.1783

2. COMPUTATIONAL EFFICIENCY (FLOPs)
Computing FLOPs...
✅ FLOPs per forward pass: 3023.66G

3. THROUGHPUT METRICS
Measuring throughput...
✅ Tokens/second: 1099.52
✅ ms/token: 0.91
✅ Samples/second: 3.87

4. PARAMETER EFFICIENCY
Analyzing parameter efficiency...
✅ Total parameters: 24.40B
✅ Active parameters: 24.40B
✅ Trainable parameters: 242.23M
✅ Sparsity ratio: 100.00%

5. MEMORY USAGE
✅ Model size: 24713.03 MB
✅ GPU allocated: 43.04 GB
✅ GPU reserved: 45.23 GB

COMBINING ALL METRICS

COMPREHENSIVE TRAINED MoE METRICS

📊 Accuracy Metrics:
  MMLU Accuracy: 0.2330
  Top-2 Accuracy: 0.4850
  ECE: 0.1783

⚡ Computational Efficiency:
  FLOPs per forward pass: 3023.66G
  Tokens/second: 1099.52
  ms/token: 0.91
  Samples/second: 3.87

🔧 Parameter Efficiency:
  Total parameters: 24.40B
  Active parameters: 24.40B
  Trainable parameters: 242.23M
  Sparsity ratio: 100.00%

💾 Memory Usage:
  Model size: 24713.03 MB
  GPU allocated: